In [2]:
# ============================================
# STEP 1 — Upload model preparation outputs
# ============================================

from google.colab import files

# Upload the ZIP file created by the
# Model Preparation notebook
uploaded = files.upload()

print("\nUpload complete!")

# Show the uploaded filename
for filename in uploaded.keys():
    print("✓", filename)

Saving medimate_model_preparation_outputs.zip to medimate_model_preparation_outputs.zip

Upload complete!
✓ medimate_model_preparation_outputs.zip


In [3]:
# ============================================
# STEP 2 — Extract model preparation package
# ============================================

import zipfile
import os

zip_path = "medimate_model_preparation_outputs.zip"
extract_dir = "medimate_model_files"

# Create the folder if it does not already exist
os.makedirs(extract_dir, exist_ok=True)

# Extract all files from the ZIP
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_dir)

print("Extraction complete!\n")

# Show all extracted files
extracted_files = sorted(os.listdir(extract_dir))

for filename in extracted_files:
    print("✓", filename)

print("\nTotal files:", len(extracted_files))

Extraction complete!

✓ X_test_no_cpra.npz
✓ X_test_with_cpra.npz
✓ X_train_no_cpra.npz
✓ X_train_with_cpra.npz
✓ X_val_no_cpra.npz
✓ X_val_with_cpra.npz
✓ features_no_cpra.csv
✓ features_with_cpra.csv
✓ preprocessor_no_cpra.joblib
✓ preprocessor_with_cpra.joblib
✓ test_metadata.csv.gz
✓ train_metadata.csv.gz
✓ val_metadata.csv.gz
✓ y_test.npy
✓ y_train.npy
✓ y_val.npy

Total files: 16


In [4]:
# ============================================
# STEP 3 — Load primary no-CPRA datasets
# ============================================

from scipy import sparse
import numpy as np
import os

base_path = "medimate_model_files"

# -------------------------
# Load processed features
# -------------------------
X_train = sparse.load_npz(
    os.path.join(base_path, "X_train_no_cpra.npz")
)

X_val = sparse.load_npz(
    os.path.join(base_path, "X_val_no_cpra.npz")
)

X_test = sparse.load_npz(
    os.path.join(base_path, "X_test_no_cpra.npz")
)

# -------------------------
# Load targets
# -------------------------
y_train = np.load(
    os.path.join(base_path, "y_train.npy")
)

y_val = np.load(
    os.path.join(base_path, "y_val.npy")
)

y_test = np.load(
    os.path.join(base_path, "y_test.npy")
)

# -------------------------
# Check shapes
# -------------------------
print("PRIMARY NO-CPRA DATA LOADED")

print("\nTRAIN")
print("X shape:", X_train.shape)
print("y shape:", y_train.shape)

print("\nVALIDATION")
print("X shape:", X_val.shape)
print("y shape:", y_val.shape)

print("\nTEST")
print("X shape:", X_test.shape)
print("y shape:", y_test.shape)

PRIMARY NO-CPRA DATA LOADED

TRAIN
X shape: (314982, 67)
y shape: (314982,)

VALIDATION
X shape: (39402, 67)
y shape: (39402,)

TEST
X shape: (41471, 67)
y shape: (41471,)


In [5]:
# ============================================
# STEP 4 — Check target balance
# ============================================

def show_target_balance(name, y):
    unique, counts = np.unique(y, return_counts=True)

    total = len(y)

    print(f"\n{name}")

    for value, count in zip(unique, counts):
        percent = count / total * 100

        print(
            f"Class {value}: {count:,} "
            f"({percent:.2f}%)"
        )


print("TARGET BALANCE")

show_target_balance("TRAIN", y_train)
show_target_balance("VALIDATION", y_val)
show_target_balance("TEST", y_test)

TARGET BALANCE

TRAIN
Class 0: 302,745 (96.12%)
Class 1: 12,237 (3.88%)

VALIDATION
Class 0: 37,956 (96.33%)
Class 1: 1,446 (3.67%)

TEST
Class 0: 39,866 (96.13%)
Class 1: 1,605 (3.87%)


In [6]:
# ============================================
# STEP 5 — Train Logistic Regression baseline
# ============================================

from sklearn.linear_model import LogisticRegression

# Create the model
# max_iter is increased so the model has enough
# iterations to finish optimization
log_reg = LogisticRegression(
    max_iter=1000,
    random_state=42
)

# Train using ONLY the training data
log_reg.fit(X_train, y_train)

print("LOGISTIC REGRESSION TRAINING COMPLETE")

LOGISTIC REGRESSION TRAINING COMPLETE


In [7]:
# ============================================
# STEP 6 — Evaluate baseline on validation set
# ============================================

from sklearn.metrics import roc_auc_score, average_precision_score

# --------------------------------------------
# Predict probability of the positive class
# Class 1 = adverse outcome within 1 year
# --------------------------------------------
val_prob = log_reg.predict_proba(X_val)[:, 1]

# --------------------------------------------
# Calculate evaluation metrics
# --------------------------------------------

# ROC-AUC:
# How well the model ranks higher-risk patients
# above lower-risk patients
val_roc_auc = roc_auc_score(
    y_val,
    val_prob
)

# PR-AUC:
# Especially important because only ~4%
# of patients have the adverse outcome
val_pr_auc = average_precision_score(
    y_val,
    val_prob
)

print("LOGISTIC REGRESSION — VALIDATION RESULTS")

print(
    "\nROC-AUC:",
    round(val_roc_auc, 4)
)

print(
    "PR-AUC:",
    round(val_pr_auc, 4)
)

LOGISTIC REGRESSION — VALIDATION RESULTS

ROC-AUC: 0.6793
PR-AUC: 0.0827


In [8]:
# ============================================
# STEP 7 — Confusion matrix at threshold 0.50
# ============================================

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score
)

# --------------------------------------------
# Convert probabilities into class predictions
#
# Probability >= 0.50 → predict Class 1
# Probability < 0.50  → predict Class 0
# --------------------------------------------
val_pred_05 = (
    val_prob >= 0.50
).astype(int)

# --------------------------------------------
# Create confusion matrix
# --------------------------------------------
cm = confusion_matrix(
    y_val,
    val_pred_05
)

tn, fp, fn, tp = cm.ravel()

print("VALIDATION — THRESHOLD 0.50")

print("\nCONFUSION MATRIX")
print(cm)

print("\nTrue Negatives :", tn)
print("False Positives:", fp)
print("False Negatives:", fn)
print("True Positives :", tp)

# --------------------------------------------
# Precision:
# Of patients predicted high-risk,
# how many actually had an adverse event?
# --------------------------------------------
precision = precision_score(
    y_val,
    val_pred_05,
    zero_division=0
)

# --------------------------------------------
# Recall:
# Of patients who actually had an adverse event,
# how many did the model catch?
# --------------------------------------------
recall = recall_score(
    y_val,
    val_pred_05,
    zero_division=0
)

print("\nPrecision:", round(precision, 4))
print("Recall:", round(recall, 4))

print("\nCLASSIFICATION REPORT")
print(
    classification_report(
        y_val,
        val_pred_05,
        digits=4,
        zero_division=0
    )
)

VALIDATION — THRESHOLD 0.50

CONFUSION MATRIX
[[37952     4]
 [ 1445     1]]

True Negatives : 37952
False Positives: 4
False Negatives: 1445
True Positives : 1

Precision: 0.2
Recall: 0.0007

CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0     0.9633    0.9999    0.9813     37956
           1     0.2000    0.0007    0.0014      1446

    accuracy                         0.9632     39402
   macro avg     0.5817    0.5003    0.4913     39402
weighted avg     0.9353    0.9632    0.9453     39402



In [9]:
# ============================================
# STEP 8 — Compare classification thresholds
# ============================================

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Thresholds we want to test
thresholds = [
    0.01,
    0.02,
    0.03,
    0.04,
    0.05,
    0.075,
    0.10,
    0.15,
    0.20,
    0.30,
    0.50
]

print("VALIDATION THRESHOLD COMPARISON\n")

for threshold in thresholds:

    # Convert predicted probabilities into
    # class predictions using this threshold
    predictions = (
        val_prob >= threshold
    ).astype(int)

    # Calculate confusion matrix
    tn, fp, fn, tp = confusion_matrix(
        y_val,
        predictions
    ).ravel()

    # Calculate important metrics
    precision = precision_score(
        y_val,
        predictions,
        zero_division=0
    )

    recall = recall_score(
        y_val,
        predictions,
        zero_division=0
    )

    f1 = f1_score(
        y_val,
        predictions,
        zero_division=0
    )

    print(
        f"Threshold {threshold:.3f} | "
        f"Precision: {precision:.3f} | "
        f"Recall: {recall:.3f} | "
        f"F1: {f1:.3f} | "
        f"TP: {tp} | "
        f"FP: {fp}"
    )

VALIDATION THRESHOLD COMPARISON

Threshold 0.010 | Precision: 0.038 | Recall: 0.992 | F1: 0.074 | TP: 1434 | FP: 36101
Threshold 0.020 | Precision: 0.044 | Recall: 0.910 | F1: 0.083 | TP: 1316 | FP: 28887
Threshold 0.030 | Precision: 0.051 | Recall: 0.777 | F1: 0.095 | TP: 1123 | FP: 21088
Threshold 0.040 | Precision: 0.062 | Recall: 0.647 | F1: 0.113 | TP: 935 | FP: 14236
Threshold 0.050 | Precision: 0.072 | Recall: 0.506 | F1: 0.127 | TP: 732 | FP: 9368
Threshold 0.075 | Precision: 0.095 | Recall: 0.216 | F1: 0.132 | TP: 313 | FP: 2971
Threshold 0.100 | Precision: 0.132 | Recall: 0.109 | F1: 0.119 | TP: 158 | FP: 1043
Threshold 0.150 | Precision: 0.185 | Recall: 0.057 | F1: 0.087 | TP: 82 | FP: 362
Threshold 0.200 | Precision: 0.212 | Recall: 0.046 | F1: 0.076 | TP: 67 | FP: 249
Threshold 0.300 | Precision: 0.220 | Recall: 0.017 | F1: 0.031 | TP: 24 | FP: 85
Threshold 0.500 | Precision: 0.200 | Recall: 0.001 | F1: 0.001 | TP: 1 | FP: 4


In [10]:
# ============================================
# STEP 9 — Find best validation threshold
# using the Precision-Recall curve
# ============================================

from sklearn.metrics import precision_recall_curve
import numpy as np

# Get precision and recall values across
# many possible probability thresholds
precision_values, recall_values, thresholds = (
    precision_recall_curve(
        y_val,
        val_prob
    )
)

# precision_recall_curve returns one extra
# precision/recall value, so match lengths
precision_for_thresholds = precision_values[:-1]
recall_for_thresholds = recall_values[:-1]

# Calculate F1 score for every threshold
f1_values = (
    2
    * precision_for_thresholds
    * recall_for_thresholds
    / (
        precision_for_thresholds
        + recall_for_thresholds
        + 1e-10
    )
)

# Find the threshold with the highest F1
best_index = np.argmax(f1_values)

best_threshold = thresholds[best_index]
best_precision = precision_for_thresholds[best_index]
best_recall = recall_for_thresholds[best_index]
best_f1 = f1_values[best_index]

print("BEST VALIDATION THRESHOLD BY F1")

print(
    "\nThreshold:",
    round(best_threshold, 4)
)

print(
    "Precision:",
    round(best_precision, 4)
)

print(
    "Recall:",
    round(best_recall, 4)
)

print(
    "F1:",
    round(best_f1, 4)
)

BEST VALIDATION THRESHOLD BY F1

Threshold: 0.0799
Precision: 0.1055
Recall: 0.1923
F1: 0.1362


In [11]:
# ============================================
# STEP 10 — Evaluate best-F1 threshold
# ============================================

from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

# Convert validation probabilities into
# class predictions using the best threshold
val_pred_best_f1 = (
    val_prob >= best_threshold
).astype(int)

# Create confusion matrix
cm_best = confusion_matrix(
    y_val,
    val_pred_best_f1
)

tn, fp, fn, tp = cm_best.ravel()

# Calculate metrics
precision_best = precision_score(
    y_val,
    val_pred_best_f1,
    zero_division=0
)

recall_best = recall_score(
    y_val,
    val_pred_best_f1,
    zero_division=0
)

f1_best = f1_score(
    y_val,
    val_pred_best_f1,
    zero_division=0
)

print("VALIDATION — BEST F1 THRESHOLD")

print(
    "\nThreshold:",
    round(best_threshold, 4)
)

print("\nCONFUSION MATRIX")
print(cm_best)

print("\nTrue Negatives :", tn)
print("False Positives:", fp)
print("False Negatives:", fn)
print("True Positives :", tp)

print(
    "\nPrecision:",
    round(precision_best, 4)
)

print(
    "Recall:",
    round(recall_best, 4)
)

print(
    "F1:",
    round(f1_best, 4)
)

VALIDATION — BEST F1 THRESHOLD

Threshold: 0.0799

CONFUSION MATRIX
[[35598  2358]
 [ 1168   278]]

True Negatives : 35598
False Positives: 2358
False Negatives: 1168
True Positives : 278

Precision: 0.1055
Recall: 0.1923
F1: 0.1362


In [12]:
# ============================================
# STEP 11 — Find threshold for >= 50% recall
# ============================================

# We want the model to catch at least
# 50% of actual adverse outcomes
TARGET_RECALL = 0.50

# Find all thresholds where recall is
# at least our target
eligible_indices = np.where(
    recall_for_thresholds >= TARGET_RECALL
)[0]

# Among those thresholds, choose the one
# with the highest precision
best_recall_index = eligible_indices[
    np.argmax(
        precision_for_thresholds[eligible_indices]
    )
]

recall_threshold = thresholds[
    best_recall_index
]

recall_point_precision = precision_for_thresholds[
    best_recall_index
]

recall_point_recall = recall_for_thresholds[
    best_recall_index
]

recall_point_f1 = f1_values[
    best_recall_index
]

print("HIGHER-RECALL VALIDATION THRESHOLD")

print(
    "\nTarget recall:",
    TARGET_RECALL
)

print(
    "Threshold:",
    round(recall_threshold, 4)
)

print(
    "Precision:",
    round(recall_point_precision, 4)
)

print(
    "Recall:",
    round(recall_point_recall, 4)
)

print(
    "F1:",
    round(recall_point_f1, 4)
)

HIGHER-RECALL VALIDATION THRESHOLD

Target recall: 0.5
Threshold: 0.0504
Precision: 0.073
Recall: 0.5021
F1: 0.1275


In [13]:
# ============================================
# STEP 12 — Evaluate higher-recall threshold
# ============================================

from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

# Convert probabilities into predictions
# using the threshold chosen for >= 50% recall
val_pred_recall = (
    val_prob >= recall_threshold
).astype(int)

# Create confusion matrix
cm_recall = confusion_matrix(
    y_val,
    val_pred_recall
)

tn, fp, fn, tp = cm_recall.ravel()

# Calculate metrics
precision_recall_point = precision_score(
    y_val,
    val_pred_recall,
    zero_division=0
)

recall_recall_point = recall_score(
    y_val,
    val_pred_recall,
    zero_division=0
)

f1_recall_point = f1_score(
    y_val,
    val_pred_recall,
    zero_division=0
)

print("VALIDATION — HIGHER-RECALL THRESHOLD")

print(
    "\nThreshold:",
    round(recall_threshold, 4)
)

print("\nCONFUSION MATRIX")
print(cm_recall)

print("\nTrue Negatives :", tn)
print("False Positives:", fp)
print("False Negatives:", fn)
print("True Positives :", tp)

print(
    "\nPrecision:",
    round(precision_recall_point, 4)
)

print(
    "Recall:",
    round(recall_recall_point, 4)
)

print(
    "F1:",
    round(f1_recall_point, 4)
)

VALIDATION — HIGHER-RECALL THRESHOLD

Threshold: 0.0504

CONFUSION MATRIX
[[28738  9218]
 [  720   726]]

True Negatives : 28738
False Positives: 9218
False Negatives: 720
True Positives : 726

Precision: 0.073
Recall: 0.5021
F1: 0.1275


In [14]:
# ============================================
# STEP 13 — Check probability calibration
# ============================================

from sklearn.metrics import brier_score_loss

# --------------------------------------------
# Brier score measures how close the predicted
# probabilities are to the actual outcomes.
#
# Lower = better
# 0 = perfect probability predictions
# --------------------------------------------

val_brier = brier_score_loss(
    y_val,
    val_prob
)

# Average actual adverse-event rate
val_event_rate = y_val.mean()

# Average risk predicted by the model
val_mean_predicted_risk = val_prob.mean()

print("VALIDATION PROBABILITY CALIBRATION")

print(
    "\nActual adverse-event rate:",
    round(val_event_rate, 4)
)

print(
    "Average predicted risk:",
    round(val_mean_predicted_risk, 4)
)

print(
    "Brier Score:",
    round(val_brier, 4)
)

VALIDATION PROBABILITY CALIBRATION

Actual adverse-event rate: 0.0367
Average predicted risk: 0.0403
Brier Score: 0.0348


In [15]:
# ============================================
# STEP 14 — Load CPRA sensitivity datasets
# ============================================

from scipy import sparse
import os

# --------------------------------------------
# Load the processed feature matrices that
# INCLUDE INIT_CPRA and INIT_CPRA_missing
# --------------------------------------------

X_train_cpra = sparse.load_npz(
    os.path.join(
        base_path,
        "X_train_with_cpra.npz"
    )
)

X_val_cpra = sparse.load_npz(
    os.path.join(
        base_path,
        "X_val_with_cpra.npz"
    )
)

X_test_cpra = sparse.load_npz(
    os.path.join(
        base_path,
        "X_test_with_cpra.npz"
    )
)

# --------------------------------------------
# Check that everything loaded correctly
# --------------------------------------------

print("CPRA SENSITIVITY DATA LOADED")

print("\nTRAIN")
print("X shape:", X_train_cpra.shape)
print("y shape:", y_train.shape)

print("\nVALIDATION")
print("X shape:", X_val_cpra.shape)
print("y shape:", y_val.shape)

print("\nTEST")
print("X shape:", X_test_cpra.shape)
print("y shape:", y_test.shape)

CPRA SENSITIVITY DATA LOADED

TRAIN
X shape: (314982, 69)
y shape: (314982,)

VALIDATION
X shape: (39402, 69)
y shape: (39402,)

TEST
X shape: (41471, 69)
y shape: (41471,)


In [16]:
# ============================================
# STEP 15 — Train Logistic Regression with CPRA
# ============================================

from sklearn.linear_model import LogisticRegression

# Create the same type of model as before
# so the comparison is fair
log_reg_cpra = LogisticRegression(
    max_iter=1000,
    random_state=42
)

# Train using the CPRA version of the
# training features
log_reg_cpra.fit(
    X_train_cpra,
    y_train
)

print(
    "LOGISTIC REGRESSION WITH CPRA "
    "TRAINING COMPLETE"
)

LOGISTIC REGRESSION WITH CPRA TRAINING COMPLETE


In [17]:
# ============================================
# STEP 16 — Evaluate CPRA model on validation
# ============================================

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss
)

# --------------------------------------------
# Predict adverse-event probabilities
# using the model WITH CPRA
# --------------------------------------------
val_prob_cpra = log_reg_cpra.predict_proba(
    X_val_cpra
)[:, 1]

# --------------------------------------------
# Calculate the same metrics used for
# the no-CPRA model
# --------------------------------------------

# Ability to rank higher-risk patients
val_roc_auc_cpra = roc_auc_score(
    y_val,
    val_prob_cpra
)

# Precision-Recall AUC
# Important because the positive class is rare
val_pr_auc_cpra = average_precision_score(
    y_val,
    val_prob_cpra
)

# Probability accuracy / calibration
# Lower Brier score is better
val_brier_cpra = brier_score_loss(
    y_val,
    val_prob_cpra
)

print("LOGISTIC REGRESSION WITH CPRA — VALIDATION")

print(
    "\nROC-AUC:",
    round(val_roc_auc_cpra, 4)
)

print(
    "PR-AUC:",
    round(val_pr_auc_cpra, 4)
)

print(
    "Brier Score:",
    round(val_brier_cpra, 4)
)

print(
    "Average predicted risk:",
    round(val_prob_cpra.mean(), 4)
)

LOGISTIC REGRESSION WITH CPRA — VALIDATION

ROC-AUC: 0.6798
PR-AUC: 0.0828
Brier Score: 0.0348
Average predicted risk: 0.0402


In [18]:
# ============================================
# STEP 17 — Compare validation performance
# NO CPRA vs WITH CPRA
# ============================================

import pandas as pd

# Create a simple comparison table
comparison = pd.DataFrame({
    "Metric": [
        "ROC-AUC",
        "PR-AUC",
        "Brier Score",
        "Average Predicted Risk"
    ],

    "No CPRA": [
        val_roc_auc,
        val_pr_auc,
        val_brier,
        val_prob.mean()
    ],

    "With CPRA": [
        val_roc_auc_cpra,
        val_pr_auc_cpra,
        val_brier_cpra,
        val_prob_cpra.mean()
    ]
})

# Calculate how much the CPRA model changed
comparison["Difference"] = (
    comparison["With CPRA"]
    - comparison["No CPRA"]
)

print("VALIDATION MODEL COMPARISON")
print(
    comparison.round(4).to_string(index=False)
)

VALIDATION MODEL COMPARISON
                Metric  No CPRA  With CPRA  Difference
               ROC-AUC   0.6793     0.6798      0.0005
                PR-AUC   0.0827     0.0828      0.0002
           Brier Score   0.0348     0.0348     -0.0000
Average Predicted Risk   0.0403     0.0402     -0.0000


In [19]:
# ============================================
# STEP 18 — Compare CPRA model thresholds
# ============================================

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Use the same thresholds as the no-CPRA model
thresholds = [
    0.01,
    0.02,
    0.03,
    0.04,
    0.05,
    0.075,
    0.10,
    0.15,
    0.20,
    0.30,
    0.50
]

print("CPRA MODEL — VALIDATION THRESHOLD COMPARISON\n")

for threshold in thresholds:

    # Turn CPRA model probabilities
    # into class predictions
    predictions = (
        val_prob_cpra >= threshold
    ).astype(int)

    # Get confusion-matrix values
    tn, fp, fn, tp = confusion_matrix(
        y_val,
        predictions
    ).ravel()

    # Calculate metrics
    precision = precision_score(
        y_val,
        predictions,
        zero_division=0
    )

    recall = recall_score(
        y_val,
        predictions,
        zero_division=0
    )

    f1 = f1_score(
        y_val,
        predictions,
        zero_division=0
    )

    print(
        f"Threshold {threshold:.3f} | "
        f"Precision: {precision:.3f} | "
        f"Recall: {recall:.3f} | "
        f"F1: {f1:.3f} | "
        f"TP: {tp} | "
        f"FP: {fp}"
    )

CPRA MODEL — VALIDATION THRESHOLD COMPARISON

Threshold 0.010 | Precision: 0.038 | Recall: 0.992 | F1: 0.073 | TP: 1434 | FP: 36161
Threshold 0.020 | Precision: 0.044 | Recall: 0.914 | F1: 0.083 | TP: 1321 | FP: 29043
Threshold 0.030 | Precision: 0.050 | Recall: 0.777 | F1: 0.094 | TP: 1124 | FP: 21222
Threshold 0.040 | Precision: 0.062 | Recall: 0.652 | F1: 0.113 | TP: 943 | FP: 14320
Threshold 0.050 | Precision: 0.072 | Recall: 0.500 | F1: 0.126 | TP: 723 | FP: 9348
Threshold 0.075 | Precision: 0.097 | Recall: 0.214 | F1: 0.133 | TP: 309 | FP: 2885
Threshold 0.100 | Precision: 0.131 | Recall: 0.104 | F1: 0.116 | TP: 151 | FP: 1002
Threshold 0.150 | Precision: 0.183 | Recall: 0.056 | F1: 0.086 | TP: 81 | FP: 361
Threshold 0.200 | Precision: 0.215 | Recall: 0.047 | F1: 0.077 | TP: 68 | FP: 249
Threshold 0.300 | Precision: 0.217 | Recall: 0.016 | F1: 0.030 | TP: 23 | FP: 83
Threshold 0.500 | Precision: 0.333 | Recall: 0.001 | F1: 0.001 | TP: 1 | FP: 2


In [20]:
# ============================================
# STEP 19 — Find best F1 threshold
# for the CPRA model
# ============================================

from sklearn.metrics import precision_recall_curve
import numpy as np

# Get precision and recall across many
# possible thresholds for the CPRA model
cpra_precision_values, cpra_recall_values, cpra_thresholds = (
    precision_recall_curve(
        y_val,
        val_prob_cpra
    )
)

# Match precision/recall lengths
# with the threshold array
cpra_precision_for_thresholds = (
    cpra_precision_values[:-1]
)

cpra_recall_for_thresholds = (
    cpra_recall_values[:-1]
)

# Calculate F1 score for every threshold
cpra_f1_values = (
    2
    * cpra_precision_for_thresholds
    * cpra_recall_for_thresholds
    / (
        cpra_precision_for_thresholds
        + cpra_recall_for_thresholds
        + 1e-10
    )
)

# Find the threshold with the highest F1
cpra_best_index = np.argmax(
    cpra_f1_values
)

cpra_best_threshold = cpra_thresholds[
    cpra_best_index
]

cpra_best_precision = (
    cpra_precision_for_thresholds[
        cpra_best_index
    ]
)

cpra_best_recall = (
    cpra_recall_for_thresholds[
        cpra_best_index
    ]
)

cpra_best_f1 = cpra_f1_values[
    cpra_best_index
]

print("CPRA MODEL — BEST VALIDATION THRESHOLD BY F1")

print(
    "\nThreshold:",
    round(cpra_best_threshold, 4)
)

print(
    "Precision:",
    round(cpra_best_precision, 4)
)

print(
    "Recall:",
    round(cpra_best_recall, 4)
)

print(
    "F1:",
    round(cpra_best_f1, 4)
)

CPRA MODEL — BEST VALIDATION THRESHOLD BY F1

Threshold: 0.0684
Precision: 0.0907
Recall: 0.2732
F1: 0.1362


In [21]:
# ============================================
# STEP 20 — Find CPRA threshold for >= 50% recall
# ============================================

# We use the same recall target as before
# so the comparison with the no-CPRA model is fair
TARGET_RECALL = 0.50

# Find thresholds where recall is at least 50%
cpra_eligible_indices = np.where(
    cpra_recall_for_thresholds >= TARGET_RECALL
)[0]

# Among those thresholds, choose the one
# with the highest precision
cpra_recall_index = cpra_eligible_indices[
    np.argmax(
        cpra_precision_for_thresholds[
            cpra_eligible_indices
        ]
    )
]

# Store the selected threshold and metrics
cpra_recall_threshold = cpra_thresholds[
    cpra_recall_index
]

cpra_recall_point_precision = (
    cpra_precision_for_thresholds[
        cpra_recall_index
    ]
)

cpra_recall_point_recall = (
    cpra_recall_for_thresholds[
        cpra_recall_index
    ]
)

cpra_recall_point_f1 = (
    cpra_f1_values[
        cpra_recall_index
    ]
)

print("CPRA MODEL — HIGHER-RECALL THRESHOLD")

print(
    "\nTarget recall:",
    TARGET_RECALL
)

print(
    "Threshold:",
    round(cpra_recall_threshold, 4)
)

print(
    "Precision:",
    round(cpra_recall_point_precision, 4)
)

print(
    "Recall:",
    round(cpra_recall_point_recall, 4)
)

print(
    "F1:",
    round(cpra_recall_point_f1, 4)
)

CPRA MODEL — HIGHER-RECALL THRESHOLD

Target recall: 0.5
Threshold: 0.0497
Precision: 0.0722
Recall: 0.5097
F1: 0.1265


In [22]:
# ============================================
# STEP 21 — Final CPRA validation comparison
# ============================================

import pandas as pd

cpra_final_comparison = pd.DataFrame({
    "Metric": [
        "ROC-AUC",
        "PR-AUC",
        "Brier Score",
        "Best F1",
        "Precision at >=50% Recall",
        "Recall at >=50% Recall",
        "F1 at >=50% Recall"
    ],

    "No CPRA": [
        val_roc_auc,
        val_pr_auc,
        val_brier,
        best_f1,
        recall_point_precision,
        recall_point_recall,
        recall_point_f1
    ],

    "With CPRA": [
        val_roc_auc_cpra,
        val_pr_auc_cpra,
        val_brier_cpra,
        cpra_best_f1,
        cpra_recall_point_precision,
        cpra_recall_point_recall,
        cpra_recall_point_f1
    ]
})

cpra_final_comparison["Difference"] = (
    cpra_final_comparison["With CPRA"]
    - cpra_final_comparison["No CPRA"]
)

print("FINAL CPRA SENSITIVITY COMPARISON — VALIDATION\n")

print(
    cpra_final_comparison
    .round(4)
    .to_string(index=False)
)

print("\nRECOMMENDATION:")
print(
    "Use the NO-CPRA feature set as the primary model. "
    "The CPRA sensitivity model provides negligible "
    "improvement in ranking metrics and no meaningful "
    "improvement in threshold-based performance."
)

FINAL CPRA SENSITIVITY COMPARISON — VALIDATION

                   Metric  No CPRA  With CPRA  Difference
                  ROC-AUC   0.6793     0.6798      0.0005
                   PR-AUC   0.0827     0.0828      0.0002
              Brier Score   0.0348     0.0348     -0.0000
                  Best F1   0.1362     0.1362     -0.0000
Precision at >=50% Recall   0.0730     0.0722     -0.0008
   Recall at >=50% Recall   0.5021     0.5097      0.0076
       F1 at >=50% Recall   0.1275     0.1265     -0.0010

RECOMMENDATION:
Use the NO-CPRA feature set as the primary model. The CPRA sensitivity model provides negligible improvement in ranking metrics and no meaningful improvement in threshold-based performance.


In [23]:
# ============================================
# STEP 22 — Calculate top-decile lift
# Primary NO-CPRA model
# ============================================

import pandas as pd
import numpy as np

# Create a validation results table
val_results = pd.DataFrame({
    "actual": y_val,
    "predicted_risk": val_prob
})

# --------------------------------------------
# Sort registrations from highest predicted
# risk to lowest predicted risk
# --------------------------------------------
val_results = val_results.sort_values(
    "predicted_risk",
    ascending=False
).reset_index(drop=True)

# --------------------------------------------
# Select the highest-risk 10%
# --------------------------------------------
top_10_count = int(
    np.ceil(len(val_results) * 0.10)
)

top_10 = val_results.iloc[
    :top_10_count
]

# Overall adverse-event rate
overall_event_rate = (
    val_results["actual"].mean()
)

# Adverse-event rate among top-risk 10%
top_10_event_rate = (
    top_10["actual"].mean()
)

# Lift = top-decile event rate
# divided by overall event rate
top_decile_lift = (
    top_10_event_rate
    / overall_event_rate
)

# How many of all adverse outcomes are
# captured inside the highest-risk 10%
top_10_capture_rate = (
    top_10["actual"].sum()
    / val_results["actual"].sum()
)

print("PRIMARY MODEL — TOP-DECILE PERFORMANCE")

print(
    "\nValidation registrations:",
    len(val_results)
)

print(
    "Top 10% registrations:",
    len(top_10)
)

print(
    "\nOverall adverse-event rate:",
    round(overall_event_rate, 4)
)

print(
    "Top 10% adverse-event rate:",
    round(top_10_event_rate, 4)
)

print(
    "Top-decile lift:",
    round(top_decile_lift, 2)
)

print(
    "Adverse outcomes captured in top 10%:",
    round(top_10_capture_rate, 4)
)

PRIMARY MODEL — TOP-DECILE PERFORMANCE

Validation registrations: 39402
Top 10% registrations: 3941

Overall adverse-event rate: 0.0367
Top 10% adverse-event rate: 0.0916
Top-decile lift: 2.5
Adverse outcomes captured in top 10%: 0.2497


In [24]:
# ============================================
# STEP 23 — Train class-weighted Logistic Regression
# Primary NO-CPRA feature set
# ============================================

from sklearn.linear_model import LogisticRegression

# --------------------------------------------
# class_weight="balanced" gives more importance
# to the rare adverse-event class (class 1)
# during training.
# --------------------------------------------

log_reg_balanced = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

# Train on the same NO-CPRA training data
log_reg_balanced.fit(
    X_train,
    y_train
)

print(
    "CLASS-WEIGHTED LOGISTIC REGRESSION "
    "TRAINING COMPLETE"
)

CLASS-WEIGHTED LOGISTIC REGRESSION TRAINING COMPLETE


In [25]:
# ============================================
# STEP 24 — Evaluate class-weighted model
# on the validation set
# ============================================

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss
)

# Predict probabilities for the adverse class
val_prob_balanced = (
    log_reg_balanced.predict_proba(X_val)[:, 1]
)

# Ranking performance
balanced_roc_auc = roc_auc_score(
    y_val,
    val_prob_balanced
)

balanced_pr_auc = average_precision_score(
    y_val,
    val_prob_balanced
)

# Probability calibration
balanced_brier = brier_score_loss(
    y_val,
    val_prob_balanced
)

print("CLASS-WEIGHTED LOGISTIC REGRESSION — VALIDATION")

print(
    "\nROC-AUC:",
    round(balanced_roc_auc, 4)
)

print(
    "PR-AUC:",
    round(balanced_pr_auc, 4)
)

print(
    "Brier Score:",
    round(balanced_brier, 4)
)

print(
    "Average predicted risk:",
    round(val_prob_balanced.mean(), 4)
)

CLASS-WEIGHTED LOGISTIC REGRESSION — VALIDATION

ROC-AUC: 0.6804
PR-AUC: 0.0822
Brier Score: 0.2293
Average predicted risk: 0.4563


In [26]:
# ============================================
# STEP 25 — Compare regular vs weighted
# Logistic Regression
# ============================================

import pandas as pd

weighted_comparison = pd.DataFrame({
    "Metric": [
        "ROC-AUC",
        "PR-AUC",
        "Brier Score",
        "Average Predicted Risk"
    ],

    "Regular Logistic Regression": [
        val_roc_auc,
        val_pr_auc,
        val_brier,
        val_prob.mean()
    ],

    "Class-Weighted Logistic Regression": [
        balanced_roc_auc,
        balanced_pr_auc,
        balanced_brier,
        val_prob_balanced.mean()
    ]
})

# Calculate the change caused by class weighting
weighted_comparison["Difference"] = (
    weighted_comparison[
        "Class-Weighted Logistic Regression"
    ]
    -
    weighted_comparison[
        "Regular Logistic Regression"
    ]
)

print("LOGISTIC REGRESSION COMPARISON — VALIDATION\n")

print(
    weighted_comparison
    .round(4)
    .to_string(index=False)
)

LOGISTIC REGRESSION COMPARISON — VALIDATION

                Metric  Regular Logistic Regression  Class-Weighted Logistic Regression  Difference
               ROC-AUC                       0.6793                              0.6804      0.0011
                PR-AUC                       0.0827                              0.0822     -0.0005
           Brier Score                       0.0348                              0.2293      0.1945
Average Predicted Risk                       0.0403                              0.4563      0.4160


In [27]:
# ============================================
# STEP 26 — Train XGBoost classifier
# Primary NO-CPRA feature set
# ============================================

from xgboost import XGBClassifier

# --------------------------------------------
# XGBoost can learn nonlinear relationships
# and interactions that Logistic Regression
# cannot capture.
#
# We start WITHOUT class weighting so that
# predicted probabilities remain easier to
# interpret and compare.
# --------------------------------------------

xgb_model = XGBClassifier(
    n_estimators=250,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

# Train on the same NO-CPRA training data
xgb_model.fit(
    X_train,
    y_train
)

print(
    "XGBOOST TRAINING COMPLETE"
)

XGBOOST TRAINING COMPLETE


In [28]:
# ============================================
# STEP 27 — Evaluate XGBoost on validation
# ============================================

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss
)

# --------------------------------------------
# Predict adverse-event probabilities
# --------------------------------------------

val_prob_xgb = xgb_model.predict_proba(
    X_val
)[:, 1]

# --------------------------------------------
# Calculate the same metrics used for
# Logistic Regression
# --------------------------------------------

xgb_roc_auc = roc_auc_score(
    y_val,
    val_prob_xgb
)

xgb_pr_auc = average_precision_score(
    y_val,
    val_prob_xgb
)

xgb_brier = brier_score_loss(
    y_val,
    val_prob_xgb
)

print("XGBOOST — VALIDATION")

print(
    "\nROC-AUC:",
    round(xgb_roc_auc, 4)
)

print(
    "PR-AUC:",
    round(xgb_pr_auc, 4)
)

print(
    "Brier Score:",
    round(xgb_brier, 4)
)

print(
    "Average predicted risk:",
    round(val_prob_xgb.mean(), 4)
)

XGBOOST — VALIDATION

ROC-AUC: 0.6855
PR-AUC: 0.085
Brier Score: 0.0347
Average predicted risk: 0.0404


In [29]:
# ============================================
# STEP 28 — Compare Logistic Regression
# vs XGBoost on validation
# ============================================

import pandas as pd

model_comparison = pd.DataFrame({
    "Metric": [
        "ROC-AUC",
        "PR-AUC",
        "Brier Score",
        "Average Predicted Risk"
    ],

    "Logistic Regression": [
        val_roc_auc,
        val_pr_auc,
        val_brier,
        val_prob.mean()
    ],

    "XGBoost": [
        xgb_roc_auc,
        xgb_pr_auc,
        xgb_brier,
        val_prob_xgb.mean()
    ]
})

# Calculate change from Logistic Regression
# to XGBoost
model_comparison["Difference"] = (
    model_comparison["XGBoost"]
    - model_comparison["Logistic Regression"]
)

print("MODEL COMPARISON — VALIDATION\n")

print(
    model_comparison
    .round(4)
    .to_string(index=False)
)

MODEL COMPARISON — VALIDATION

                Metric  Logistic Regression  XGBoost  Difference
               ROC-AUC               0.6793   0.6855      0.0062
                PR-AUC               0.0827   0.0850      0.0024
           Brier Score               0.0348   0.0347     -0.0000
Average Predicted Risk               0.0403   0.0404      0.0001


In [30]:
# ============================================
# STEP 29 — Find best F1 threshold
# for XGBoost
# ============================================

from sklearn.metrics import precision_recall_curve
import numpy as np

# Get precision and recall across many
# possible XGBoost probability thresholds
xgb_precision_values, xgb_recall_values, xgb_thresholds = (
    precision_recall_curve(
        y_val,
        val_prob_xgb
    )
)

# Match precision/recall arrays
# with the threshold array
xgb_precision_for_thresholds = (
    xgb_precision_values[:-1]
)

xgb_recall_for_thresholds = (
    xgb_recall_values[:-1]
)

# Calculate F1 score at every threshold
xgb_f1_values = (
    2
    * xgb_precision_for_thresholds
    * xgb_recall_for_thresholds
    / (
        xgb_precision_for_thresholds
        + xgb_recall_for_thresholds
        + 1e-10
    )
)

# Find threshold with highest F1
xgb_best_index = np.argmax(
    xgb_f1_values
)

xgb_best_threshold = xgb_thresholds[
    xgb_best_index
]

xgb_best_precision = (
    xgb_precision_for_thresholds[
        xgb_best_index
    ]
)

xgb_best_recall = (
    xgb_recall_for_thresholds[
        xgb_best_index
    ]
)

xgb_best_f1 = xgb_f1_values[
    xgb_best_index
]

print("XGBOOST — BEST VALIDATION THRESHOLD BY F1")

print(
    "\nThreshold:",
    round(xgb_best_threshold, 4)
)

print(
    "Precision:",
    round(xgb_best_precision, 4)
)

print(
    "Recall:",
    round(xgb_best_recall, 4)
)

print(
    "F1:",
    round(xgb_best_f1, 4)
)

XGBOOST — BEST VALIDATION THRESHOLD BY F1

Threshold: 0.0667
Precision: 0.0906
Recall: 0.2863
F1: 0.1377


In [31]:
# ============================================
# STEP 30 — Find XGBoost threshold
# for >= 50% recall
# ============================================

TARGET_RECALL = 0.50

# Find all thresholds where XGBoost
# reaches at least 50% recall
xgb_eligible_indices = np.where(
    xgb_recall_for_thresholds >= TARGET_RECALL
)[0]

# Among those thresholds, choose the one
# with the highest precision
xgb_recall_index = xgb_eligible_indices[
    np.argmax(
        xgb_precision_for_thresholds[
            xgb_eligible_indices
        ]
    )
]

# Save the selected threshold and metrics
xgb_recall_threshold = xgb_thresholds[
    xgb_recall_index
]

xgb_recall_point_precision = (
    xgb_precision_for_thresholds[
        xgb_recall_index
    ]
)

xgb_recall_point_recall = (
    xgb_recall_for_thresholds[
        xgb_recall_index
    ]
)

xgb_recall_point_f1 = (
    xgb_f1_values[
        xgb_recall_index
    ]
)

print("XGBOOST — HIGHER-RECALL THRESHOLD")

print(
    "\nTarget recall:",
    TARGET_RECALL
)

print(
    "Threshold:",
    round(xgb_recall_threshold, 4)
)

print(
    "Precision:",
    round(xgb_recall_point_precision, 4)
)

print(
    "Recall:",
    round(xgb_recall_point_recall, 4)
)

print(
    "F1:",
    round(xgb_recall_point_f1, 4)
)

XGBOOST — HIGHER-RECALL THRESHOLD

Target recall: 0.5
Threshold: 0.051
Precision: 0.0737
Recall: 0.5
F1: 0.1284


In [32]:
# ============================================
# STEP 31 — XGBoost top-decile performance
# ============================================

import pandas as pd
import numpy as np

# Create validation table
xgb_val_results = pd.DataFrame({
    "actual": y_val,
    "predicted_risk": val_prob_xgb
})

# Sort from highest predicted risk
# to lowest predicted risk
xgb_val_results = xgb_val_results.sort_values(
    "predicted_risk",
    ascending=False
).reset_index(drop=True)

# Select highest-risk 10%
xgb_top_10_count = int(
    np.ceil(len(xgb_val_results) * 0.10)
)

xgb_top_10 = xgb_val_results.iloc[
    :xgb_top_10_count
]

# Overall adverse-event rate
xgb_overall_event_rate = (
    xgb_val_results["actual"].mean()
)

# Event rate inside highest-risk 10%
xgb_top_10_event_rate = (
    xgb_top_10["actual"].mean()
)

# Calculate lift
xgb_top_decile_lift = (
    xgb_top_10_event_rate
    / xgb_overall_event_rate
)

# Percentage of all adverse outcomes
# found inside the highest-risk 10%
xgb_top_10_capture_rate = (
    xgb_top_10["actual"].sum()
    / xgb_val_results["actual"].sum()
)

print("XGBOOST — TOP-DECILE PERFORMANCE")

print(
    "\nOverall adverse-event rate:",
    round(xgb_overall_event_rate, 4)
)

print(
    "Top 10% adverse-event rate:",
    round(xgb_top_10_event_rate, 4)
)

print(
    "Top-decile lift:",
    round(xgb_top_decile_lift, 2)
)

print(
    "Adverse outcomes captured in top 10%:",
    round(xgb_top_10_capture_rate, 4)
)

XGBOOST — TOP-DECILE PERFORMANCE

Overall adverse-event rate: 0.0367
Top 10% adverse-event rate: 0.0931
Top-decile lift: 2.54
Adverse outcomes captured in top 10%: 0.2538


In [33]:
# ============================================
# STEP 32 — Final validation model comparison
# ============================================

import pandas as pd

final_model_comparison = pd.DataFrame({
    "Metric": [
        "ROC-AUC",
        "PR-AUC",
        "Brier Score",
        "Average Predicted Risk",
        "Best F1",
        "Precision at >=50% Recall",
        "F1 at >=50% Recall",
        "Top-Decile Lift",
        "Top 10% Adverse Capture Rate"
    ],

    "Logistic Regression": [
        val_roc_auc,
        val_pr_auc,
        val_brier,
        val_prob.mean(),
        best_f1,
        recall_point_precision,
        recall_point_f1,
        top_decile_lift,
        top_10_capture_rate
    ],

    "XGBoost": [
        xgb_roc_auc,
        xgb_pr_auc,
        xgb_brier,
        val_prob_xgb.mean(),
        xgb_best_f1,
        xgb_recall_point_precision,
        xgb_recall_point_f1,
        xgb_top_decile_lift,
        xgb_top_10_capture_rate
    ]
})

# Difference = XGBoost minus Logistic Regression
final_model_comparison["Difference"] = (
    final_model_comparison["XGBoost"]
    - final_model_comparison["Logistic Regression"]
)

print("FINAL MODEL COMPARISON — VALIDATION\n")

print(
    final_model_comparison
    .round(4)
    .to_string(index=False)
)

print("\nCURRENT LEADING MODEL:")
print(
    "XGBoost — it shows modest but consistent "
    "improvement over Logistic Regression."
)

FINAL MODEL COMPARISON — VALIDATION

                      Metric  Logistic Regression  XGBoost  Difference
                     ROC-AUC               0.6793   0.6855      0.0062
                      PR-AUC               0.0827   0.0850      0.0024
                 Brier Score               0.0348   0.0347     -0.0000
      Average Predicted Risk               0.0403   0.0404      0.0001
                     Best F1               0.1362   0.1377      0.0015
   Precision at >=50% Recall               0.0730   0.0737      0.0007
          F1 at >=50% Recall               0.1275   0.1284      0.0010
             Top-Decile Lift               2.4960   2.5375      0.0415
Top 10% Adverse Capture Rate               0.2497   0.2538      0.0041

CURRENT LEADING MODEL:
XGBoost — it shows modest but consistent improvement over Logistic Regression.


In [34]:
# ============================================
# STEP 33 — Small XGBoost tuning experiment
# Validation set only
# ============================================

from xgboost import XGBClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss
)
import pandas as pd

# --------------------------------------------
# Try only a few reasonable configurations.
# Keeping this small reduces overfitting to
# the validation set.
# --------------------------------------------

xgb_configs = [
    {
        "name": "Current",
        "n_estimators": 250,
        "max_depth": 4,
        "learning_rate": 0.05
    },
    {
        "name": "Shallower",
        "n_estimators": 400,
        "max_depth": 3,
        "learning_rate": 0.03
    },
    {
        "name": "Deeper",
        "n_estimators": 300,
        "max_depth": 5,
        "learning_rate": 0.05
    },
    {
        "name": "Faster",
        "n_estimators": 200,
        "max_depth": 3,
        "learning_rate": 0.08
    }
]

tuning_results = []
trained_xgb_models = {}

# --------------------------------------------
# Train each model and evaluate ONLY on
# the validation set
# --------------------------------------------

for config in xgb_configs:

    model = XGBClassifier(
        n_estimators=config["n_estimators"],
        max_depth=config["max_depth"],
        learning_rate=config["learning_rate"],
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train,
        y_train
    )

    val_prob_temp = model.predict_proba(
        X_val
    )[:, 1]

    roc = roc_auc_score(
        y_val,
        val_prob_temp
    )

    pr = average_precision_score(
        y_val,
        val_prob_temp
    )

    brier = brier_score_loss(
        y_val,
        val_prob_temp
    )

    tuning_results.append({
        "Model": config["name"],
        "Trees": config["n_estimators"],
        "Depth": config["max_depth"],
        "Learning Rate": config["learning_rate"],
        "ROC-AUC": roc,
        "PR-AUC": pr,
        "Brier": brier
    })

    trained_xgb_models[
        config["name"]
    ] = model


# --------------------------------------------
# Put results in a table
# and rank primarily by PR-AUC
# --------------------------------------------

tuning_results_df = pd.DataFrame(
    tuning_results
)

tuning_results_df = (
    tuning_results_df
    .sort_values(
        "PR-AUC",
        ascending=False
    )
    .reset_index(drop=True)
)

print("XGBOOST TUNING — VALIDATION\n")

print(
    tuning_results_df
    .round(4)
    .to_string(index=False)
)

XGBOOST TUNING — VALIDATION

    Model  Trees  Depth  Learning Rate  ROC-AUC  PR-AUC  Brier
   Deeper    300      5           0.05   0.6866  0.0853 0.0348
  Current    250      4           0.05   0.6855  0.0850 0.0347
Shallower    400      3           0.03   0.6832  0.0843 0.0347
   Faster    200      3           0.08   0.6845  0.0840 0.0347


In [35]:
# ============================================
# STEP 34 — Evaluate tuned Deeper XGBoost
# ============================================

from sklearn.metrics import precision_recall_curve
import numpy as np
import pandas as pd

# Get the already-trained Deeper model
xgb_deeper = trained_xgb_models["Deeper"]

# Predict validation probabilities
val_prob_xgb_deeper = xgb_deeper.predict_proba(
    X_val
)[:, 1]

# --------------------------------------------
# 1. Find best F1 threshold
# --------------------------------------------

deeper_precision, deeper_recall, deeper_thresholds = (
    precision_recall_curve(
        y_val,
        val_prob_xgb_deeper
    )
)

deeper_precision_t = deeper_precision[:-1]
deeper_recall_t = deeper_recall[:-1]

deeper_f1_values = (
    2 * deeper_precision_t * deeper_recall_t
    / (
        deeper_precision_t
        + deeper_recall_t
        + 1e-10
    )
)

deeper_best_index = np.argmax(
    deeper_f1_values
)

deeper_best_threshold = deeper_thresholds[
    deeper_best_index
]

deeper_best_precision = deeper_precision_t[
    deeper_best_index
]

deeper_best_recall = deeper_recall_t[
    deeper_best_index
]

deeper_best_f1 = deeper_f1_values[
    deeper_best_index
]

# --------------------------------------------
# 2. Find threshold for >= 50% recall
# --------------------------------------------

TARGET_RECALL = 0.50

eligible = np.where(
    deeper_recall_t >= TARGET_RECALL
)[0]

deeper_recall_index = eligible[
    np.argmax(
        deeper_precision_t[eligible]
    )
]

deeper_50_threshold = deeper_thresholds[
    deeper_recall_index
]

deeper_50_precision = deeper_precision_t[
    deeper_recall_index
]

deeper_50_recall = deeper_recall_t[
    deeper_recall_index
]

deeper_50_f1 = deeper_f1_values[
    deeper_recall_index
]

# --------------------------------------------
# 3. Calculate top-decile lift
# --------------------------------------------

deeper_results = pd.DataFrame({
    "actual": y_val,
    "predicted_risk": val_prob_xgb_deeper
}).sort_values(
    "predicted_risk",
    ascending=False
).reset_index(drop=True)

top_n = int(
    np.ceil(len(deeper_results) * 0.10)
)

deeper_top_10 = deeper_results.iloc[:top_n]

deeper_event_rate = deeper_results[
    "actual"
].mean()

deeper_top_10_event_rate = deeper_top_10[
    "actual"
].mean()

deeper_top_decile_lift = (
    deeper_top_10_event_rate
    / deeper_event_rate
)

deeper_capture_rate = (
    deeper_top_10["actual"].sum()
    / deeper_results["actual"].sum()
)

# --------------------------------------------
# Print results
# --------------------------------------------

print("DEEPER XGBOOST — VALIDATION PERFORMANCE")

print("\nBEST F1")
print("Threshold:", round(deeper_best_threshold, 4))
print("Precision:", round(deeper_best_precision, 4))
print("Recall:", round(deeper_best_recall, 4))
print("F1:", round(deeper_best_f1, 4))

print("\nAT >= 50% RECALL")
print("Threshold:", round(deeper_50_threshold, 4))
print("Precision:", round(deeper_50_precision, 4))
print("Recall:", round(deeper_50_recall, 4))
print("F1:", round(deeper_50_f1, 4))

print("\nTOP-DECILE PERFORMANCE")
print(
    "Top-decile lift:",
    round(deeper_top_decile_lift, 2)
)
print(
    "Adverse outcomes captured:",
    round(deeper_capture_rate, 4)
)

DEEPER XGBOOST — VALIDATION PERFORMANCE

BEST F1
Threshold: 0.0769
Precision: 0.1048
Recall: 0.2172
F1: 0.1414

AT >= 50% RECALL
Threshold: 0.0506
Precision: 0.0727
Recall: 0.5007
F1: 0.127

TOP-DECILE PERFORMANCE
Top-decile lift: 2.52
Adverse outcomes captured: 0.2524


In [36]:
# ============================================
# STEP 35 — Compare Current vs Deeper XGBoost
# ============================================

import pandas as pd

xgb_tuned_comparison = pd.DataFrame({
    "Metric": [
        "ROC-AUC",
        "PR-AUC",
        "Brier Score",
        "Best F1",
        "Best-F1 Precision",
        "Best-F1 Recall",
        "Precision at >=50% Recall",
        "F1 at >=50% Recall",
        "Top-Decile Lift",
        "Top 10% Adverse Capture Rate"
    ],

    "Current XGBoost": [
        xgb_roc_auc,
        xgb_pr_auc,
        xgb_brier,
        xgb_best_f1,
        xgb_best_precision,
        xgb_best_recall,
        xgb_recall_point_precision,
        xgb_recall_point_f1,
        xgb_top_decile_lift,
        xgb_top_10_capture_rate
    ],

    "Deeper XGBoost": [
        0.6866,
        0.0853,
        0.0348,
        deeper_best_f1,
        deeper_best_precision,
        deeper_best_recall,
        deeper_50_precision,
        deeper_50_f1,
        deeper_top_decile_lift,
        deeper_capture_rate
    ]
})

xgb_tuned_comparison["Difference"] = (
    xgb_tuned_comparison["Deeper XGBoost"]
    - xgb_tuned_comparison["Current XGBoost"]
)

print("XGBOOST MODEL SELECTION — VALIDATION\n")

print(
    xgb_tuned_comparison
    .round(4)
    .to_string(index=False)
)

XGBOOST MODEL SELECTION — VALIDATION

                      Metric  Current XGBoost  Deeper XGBoost  Difference
                     ROC-AUC           0.6855          0.6866      0.0011
                      PR-AUC           0.0850          0.0853      0.0003
                 Brier Score           0.0347          0.0348      0.0001
                     Best F1           0.1377          0.1414      0.0037
           Best-F1 Precision           0.0906          0.1048      0.0142
              Best-F1 Recall           0.2863          0.2172     -0.0692
   Precision at >=50% Recall           0.0737          0.0727     -0.0010
          F1 at >=50% Recall           0.1284          0.1270     -0.0014
             Top-Decile Lift           2.5375          2.5237     -0.0138
Top 10% Adverse Capture Rate           0.2538          0.2524     -0.0014


In [37]:
# ============================================
# STEP 36 — FINAL evaluation on 2024 test set
# ============================================

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# --------------------------------------------
# IMPORTANT:
# We are using the CURRENT XGBoost model.
#
# We do NOT tune anything using the test set.
# Thresholds were already chosen using
# validation data.
# --------------------------------------------

# Predict probabilities on untouched 2024 test data
test_prob_xgb = xgb_model.predict_proba(
    X_test
)[:, 1]

# --------------------------------------------
# Threshold-free metrics
# --------------------------------------------

test_roc_auc = roc_auc_score(
    y_test,
    test_prob_xgb
)

test_pr_auc = average_precision_score(
    y_test,
    test_prob_xgb
)

test_brier = brier_score_loss(
    y_test,
    test_prob_xgb
)

# --------------------------------------------
# Evaluate using the validation-derived
# >=50% recall threshold
#
# DO NOT find a new threshold on test data.
# --------------------------------------------

test_pred_50 = (
    test_prob_xgb >= xgb_recall_threshold
).astype(int)

tn, fp, fn, tp = confusion_matrix(
    y_test,
    test_pred_50
).ravel()

test_precision_50 = precision_score(
    y_test,
    test_pred_50,
    zero_division=0
)

test_recall_50 = recall_score(
    y_test,
    test_pred_50,
    zero_division=0
)

test_f1_50 = f1_score(
    y_test,
    test_pred_50,
    zero_division=0
)

print("FINAL XGBOOST — 2024 TEST SET")

print("\nTHRESHOLD-FREE METRICS")
print("ROC-AUC:", round(test_roc_auc, 4))
print("PR-AUC:", round(test_pr_auc, 4))
print("Brier Score:", round(test_brier, 4))

print(
    "Average predicted risk:",
    round(test_prob_xgb.mean(), 4)
)

print(
    "Actual adverse-event rate:",
    round(y_test.mean(), 4)
)

print("\nVALIDATION-DERIVED >=50% RECALL THRESHOLD")
print(
    "Threshold:",
    round(xgb_recall_threshold, 4)
)

print("\nCONFUSION MATRIX")
print(
    confusion_matrix(
        y_test,
        test_pred_50
    )
)

print("\nTrue Negatives :", tn)
print("False Positives:", fp)
print("False Negatives:", fn)
print("True Positives :", tp)

print(
    "\nPrecision:",
    round(test_precision_50, 4)
)

print(
    "Recall:",
    round(test_recall_50, 4)
)

print(
    "F1:",
    round(test_f1_50, 4)
)

FINAL XGBOOST — 2024 TEST SET

THRESHOLD-FREE METRICS
ROC-AUC: 0.6742
PR-AUC: 0.086
Brier Score: 0.0366
Average predicted risk: 0.0408
Actual adverse-event rate: 0.0387

VALIDATION-DERIVED >=50% RECALL THRESHOLD
Threshold: 0.051

CONFUSION MATRIX
[[29867  9999]
 [  831   774]]

True Negatives : 29867
False Positives: 9999
False Negatives: 831
True Positives : 774

Precision: 0.0718
Recall: 0.4822
F1: 0.1251


In [38]:
# ============================================
# STEP 37 — Final test-set top-decile performance
# ============================================

import pandas as pd
import numpy as np

# Create test results table
test_results = pd.DataFrame({
    "actual": y_test,
    "predicted_risk": test_prob_xgb
})

# Sort registrations from highest predicted
# risk to lowest predicted risk
test_results = test_results.sort_values(
    "predicted_risk",
    ascending=False
).reset_index(drop=True)

# Select the highest-risk 10%
test_top_10_count = int(
    np.ceil(len(test_results) * 0.10)
)

test_top_10 = test_results.iloc[
    :test_top_10_count
]

# Overall adverse-event rate
test_overall_event_rate = (
    test_results["actual"].mean()
)

# Adverse-event rate in highest-risk 10%
test_top_10_event_rate = (
    test_top_10["actual"].mean()
)

# Top-decile lift
test_top_decile_lift = (
    test_top_10_event_rate
    / test_overall_event_rate
)

# Fraction of ALL adverse outcomes captured
# inside the highest-risk 10%
test_top_10_capture_rate = (
    test_top_10["actual"].sum()
    / test_results["actual"].sum()
)

print("FINAL XGBOOST — TEST TOP-DECILE PERFORMANCE")

print(
    "\nTest registrations:",
    len(test_results)
)

print(
    "Top 10% registrations:",
    len(test_top_10)
)

print(
    "\nOverall adverse-event rate:",
    round(test_overall_event_rate, 4)
)

print(
    "Top 10% adverse-event rate:",
    round(test_top_10_event_rate, 4)
)

print(
    "Top-decile lift:",
    round(test_top_decile_lift, 2)
)

print(
    "Adverse outcomes captured in top 10%:",
    round(test_top_10_capture_rate, 4)
)

FINAL XGBOOST — TEST TOP-DECILE PERFORMANCE

Test registrations: 41471
Top 10% registrations: 4148

Overall adverse-event rate: 0.0387
Top 10% adverse-event rate: 0.0979
Top-decile lift: 2.53
Adverse outcomes captured in top 10%: 0.253


In [39]:
# ============================================
# STEP 38 — Validation vs Test summary
# Final XGBoost model
# ============================================

import pandas as pd

final_generalization_table = pd.DataFrame({
    "Metric": [
        "ROC-AUC",
        "PR-AUC",
        "Brier Score",
        "Average Predicted Risk",
        "Actual Adverse Rate",
        "Precision at Fixed Threshold",
        "Recall at Fixed Threshold",
        "F1 at Fixed Threshold",
        "Top-Decile Lift",
        "Top 10% Adverse Capture Rate"
    ],

    "Validation 2023": [
        xgb_roc_auc,
        xgb_pr_auc,
        xgb_brier,
        val_prob_xgb.mean(),
        y_val.mean(),
        xgb_recall_point_precision,
        xgb_recall_point_recall,
        xgb_recall_point_f1,
        xgb_top_decile_lift,
        xgb_top_10_capture_rate
    ],

    "Test 2024": [
        test_roc_auc,
        test_pr_auc,
        test_brier,
        test_prob_xgb.mean(),
        y_test.mean(),
        test_precision_50,
        test_recall_50,
        test_f1_50,
        test_top_decile_lift,
        test_top_10_capture_rate
    ]
})

final_generalization_table["Difference"] = (
    final_generalization_table["Test 2024"]
    - final_generalization_table["Validation 2023"]
)

print("FINAL XGBOOST — GENERALIZATION SUMMARY\n")

print(
    final_generalization_table
    .round(4)
    .to_string(index=False)
)

FINAL XGBOOST — GENERALIZATION SUMMARY

                      Metric  Validation 2023  Test 2024  Difference
                     ROC-AUC           0.6855     0.6742     -0.0113
                      PR-AUC           0.0850     0.0860      0.0009
                 Brier Score           0.0347     0.0366      0.0018
      Average Predicted Risk           0.0404     0.0408      0.0004
         Actual Adverse Rate           0.0367     0.0387      0.0020
Precision at Fixed Threshold           0.0737     0.0718     -0.0018
   Recall at Fixed Threshold           0.5000     0.4822     -0.0178
       F1 at Fixed Threshold           0.1284     0.1251     -0.0034
             Top-Decile Lift           2.5375     2.5290     -0.0085
Top 10% Adverse Capture Rate           0.2538     0.2530     -0.0008


In [40]:
# ============================================
# STEP 39 — Load test metadata for
# subgroup / fairness evaluation
# ============================================

import pandas as pd
import os
import numpy as np

# Load metadata that was saved during
# Model Preparation
test_metadata = pd.read_csv(
    os.path.join(
        base_path,
        "test_metadata.csv.gz"
    )
)

print("TEST METADATA LOADED")

print(
    "\nMetadata shape:",
    test_metadata.shape
)

print(
    "Prediction count:",
    len(test_prob_xgb)
)

print(
    "Target count:",
    len(y_test)
)

# --------------------------------------------
# Verify that the saved target column in
# metadata matches y_test exactly.
#
# This is important before calculating
# subgroup metrics.
# --------------------------------------------

target_alignment = np.array_equal(
    test_metadata["adverse_1yr"].to_numpy(),
    y_test
)

print(
    "\nTarget alignment:",
    target_alignment
)

# Show available subgroup columns
print("\nAVAILABLE METADATA COLUMNS")

print(
    test_metadata.columns.tolist()
)

TEST METADATA LOADED

Metadata shape: (41471, 11)
Prediction count: 41471
Target count: 41471

Target alignment: True

AVAILABLE METADATA COLUMNS
['PT_CODE', 'WL_ID_CODE', 'INIT_DATE', 'listing_year', 'LISTING_CTR_CODE', 'INIT_AGE', 'GENDER', 'ETHCAT', 'ABO', 'REGION', 'adverse_1yr']


In [41]:
# ============================================
# STEP 40 — Subgroup performance by GENDER
# Final 2024 test set
# ============================================

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    precision_score,
    recall_score,
    f1_score
)

import pandas as pd
import numpy as np


# --------------------------------------------
# Add final model predictions to the
# aligned test metadata
# --------------------------------------------

fairness_test = test_metadata.copy()

fairness_test["predicted_risk"] = test_prob_xgb

fairness_test["prediction"] = (
    test_prob_xgb >= xgb_recall_threshold
).astype(int)


# --------------------------------------------
# Function for calculating performance
# separately for each subgroup
# --------------------------------------------

def subgroup_metrics(data, group_column):

    rows = []

    for group, group_df in data.groupby(
        group_column,
        dropna=False
    ):

        y_true = group_df["adverse_1yr"]
        y_prob = group_df["predicted_risk"]
        y_pred = group_df["prediction"]

        # ROC-AUC requires both classes
        if y_true.nunique() == 2:
            roc_auc = roc_auc_score(
                y_true,
                y_prob
            )

            pr_auc = average_precision_score(
                y_true,
                y_prob
            )
        else:
            roc_auc = np.nan
            pr_auc = np.nan

        rows.append({
            group_column: group,
            "N": len(group_df),
            "Adverse Rate": y_true.mean(),
            "Average Predicted Risk": y_prob.mean(),
            "ROC-AUC": roc_auc,
            "PR-AUC": pr_auc,
            "Brier Score": brier_score_loss(
                y_true,
                y_prob
            ),
            "Precision": precision_score(
                y_true,
                y_pred,
                zero_division=0
            ),
            "Recall": recall_score(
                y_true,
                y_pred,
                zero_division=0
            ),
            "F1": f1_score(
                y_true,
                y_pred,
                zero_division=0
            )
        })

    return pd.DataFrame(rows)


# --------------------------------------------
# Calculate gender subgroup results
# --------------------------------------------

gender_results = subgroup_metrics(
    fairness_test,
    "GENDER"
)

print("FINAL XGBOOST — TEST PERFORMANCE BY GENDER\n")

print(
    gender_results
    .round(4)
    .to_string(index=False)
)

FINAL XGBOOST — TEST PERFORMANCE BY GENDER

GENDER     N  Adverse Rate  Average Predicted Risk  ROC-AUC  PR-AUC  Brier Score  Precision  Recall     F1
     F 15830        0.0371                  0.0399   0.6832  0.0938       0.0349     0.0722  0.4787 0.1255
     M 25641        0.0397                  0.0413   0.6685  0.0823       0.0376     0.0716  0.4843 0.1248


In [42]:
# ============================================
# STEP 41 — Subgroup performance by
# race / ethnicity (ETHCAT)
# ============================================

# Reuse the subgroup_metrics() function
# created in Step 40

ethnicity_results = subgroup_metrics(
    fairness_test,
    "ETHCAT"
)

# Sort by group size so the largest groups
# appear first
ethnicity_results = ethnicity_results.sort_values(
    "N",
    ascending=False
).reset_index(drop=True)

print(
    "FINAL XGBOOST — TEST PERFORMANCE "
    "BY RACE / ETHNICITY\n"
)

print(
    ethnicity_results
    .round(4)
    .to_string(index=False)
)

FINAL XGBOOST — TEST PERFORMANCE BY RACE / ETHNICITY

 ETHCAT     N  Adverse Rate  Average Predicted Risk  ROC-AUC  PR-AUC  Brier Score  Precision  Recall     F1
      1 15984        0.0444                  0.0467   0.6801  0.0917       0.0417     0.0766  0.5972 0.1358
      2 12080        0.0378                  0.0382   0.6370  0.0667       0.0361     0.0626  0.3611 0.1067
      4  8985        0.0343                  0.0372   0.6981  0.1139       0.0320     0.0760  0.4708 0.1308
      5  3036        0.0287                  0.0318   0.6217  0.0654       0.0276     0.0533  0.2414 0.0873
    998   433        0.0416                  0.0383   0.6660  0.1168       0.0398     0.0729  0.3889 0.1228
      9   399        0.0150                  0.0323   0.7481  0.0345       0.0153     0.0290  0.3333 0.0533
      6   344        0.0233                  0.0391   0.6838  0.1103       0.0223     0.0395  0.3750 0.0714
      7   210        0.0524                  0.0429   0.7387  0.1991       0.0475 

In [43]:
# ============================================
# STEP 42 — Subgroup performance by ABO
# blood type
# ============================================

# Reuse the subgroup_metrics() function
# from Step 40

abo_results = subgroup_metrics(
    fairness_test,
    "ABO"
)

# Sort by group size
abo_results = abo_results.sort_values(
    "N",
    ascending=False
).reset_index(drop=True)

print(
    "FINAL XGBOOST — TEST PERFORMANCE "
    "BY BLOOD TYPE\n"
)

print(
    abo_results
    .round(4)
    .to_string(index=False)
)

FINAL XGBOOST — TEST PERFORMANCE BY BLOOD TYPE

ABO     N  Adverse Rate  Average Predicted Risk  ROC-AUC  PR-AUC  Brier Score  Precision  Recall     F1
  O 20372        0.0385                  0.0408   0.6753  0.0862       0.0364     0.0707  0.4707 0.1229
  A 13167        0.0405                  0.0421   0.6861  0.1012       0.0379     0.0772  0.5272 0.1347
  B  6116        0.0345                  0.0386   0.6499  0.0615       0.0333     0.0612  0.4360 0.1074
 AB  1606        0.0436                  0.0379   0.6635  0.0898       0.0411     0.0797  0.4143 0.1336
 A1   160        0.0250                  0.0350   0.4487  0.1041       0.0246     0.0385  0.2500 0.0667
 A2    29        0.0345                  0.0459   0.7500  0.1250       0.0328     0.0909  1.0000 0.1667
A1B    15        0.1333                  0.0496   0.4231  0.1769       0.1261     0.1429  0.5000 0.2222
A2B     6        0.0000                  0.0372      NaN     NaN       0.0020     0.0000  0.0000 0.0000


In [44]:
# ============================================
# STEP 43 — Subgroup performance by AGE GROUP
# ============================================

# Create age groups from INIT_AGE
fairness_test["AGE_GROUP"] = pd.cut(
    fairness_test["INIT_AGE"],
    bins=[
        -np.inf,
        17,
        34,
        49,
        64,
        np.inf
    ],
    labels=[
        "0-17",
        "18-34",
        "35-49",
        "50-64",
        "65+"
    ]
)

# Calculate subgroup performance
age_results = subgroup_metrics(
    fairness_test,
    "AGE_GROUP"
)

# Keep the natural age order
age_order = [
    "0-17",
    "18-34",
    "35-49",
    "50-64",
    "65+"
]

age_results["AGE_GROUP"] = pd.Categorical(
    age_results["AGE_GROUP"],
    categories=age_order,
    ordered=True
)

age_results = age_results.sort_values(
    "AGE_GROUP"
).reset_index(drop=True)

print(
    "FINAL XGBOOST — TEST PERFORMANCE "
    "BY AGE GROUP\n"
)

print(
    age_results
    .round(4)
    .to_string(index=False)
)

/tmp/ipykernel_1087/3391814255.py:42: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for group, group_df in data.groupby(


FINAL XGBOOST — TEST PERFORMANCE BY AGE GROUP

AGE_GROUP     N  Adverse Rate  Average Predicted Risk  ROC-AUC  PR-AUC  Brier Score  Precision  Recall     F1
     0-17  1091        0.0128                  0.0108   0.8353  0.0739       0.0124     0.0833  0.0714 0.0769
    18-34  3806        0.0150                  0.0153   0.6209  0.0410       0.0146     0.1220  0.0877 0.1020
    35-49 10089        0.0241                  0.0249   0.6355  0.0695       0.0231     0.1229  0.1193 0.1211
    50-64 16722        0.0437                  0.0440   0.6210  0.0812       0.0413     0.0707  0.3986 0.1200
      65+  9763        0.0575                  0.0650   0.6239  0.0995       0.0536     0.0704  0.7986 0.1293


In [45]:
# ============================================
# STEP 44 — Subgroup performance by REGION
# ============================================

# Reuse the subgroup_metrics() function
# created earlier

region_results = subgroup_metrics(
    fairness_test,
    "REGION"
)

# Sort regions numerically
region_results = region_results.sort_values(
    "REGION"
).reset_index(drop=True)

print(
    "FINAL XGBOOST — TEST PERFORMANCE "
    "BY OPTN REGION\n"
)

print(
    region_results
    .round(4)
    .to_string(index=False)
)

FINAL XGBOOST — TEST PERFORMANCE BY OPTN REGION

 REGION    N  Adverse Rate  Average Predicted Risk  ROC-AUC  PR-AUC  Brier Score  Precision  Recall     F1
      1 1580        0.0285                  0.0410   0.7409  0.1056       0.0271     0.0691  0.6222 0.1244
      2 4482        0.0406                  0.0414   0.6916  0.0883       0.0383     0.0785  0.5440 0.1372
      3 5691        0.0467                  0.0411   0.6629  0.1005       0.0437     0.0837  0.4662 0.1419
      4 4607        0.0486                  0.0496   0.7040  0.1361       0.0446     0.0856  0.6205 0.1504
      5 7011        0.0301                  0.0388   0.6610  0.0592       0.0292     0.0565  0.4313 0.0998
      6 1182        0.0372                  0.0354   0.6910  0.0858       0.0352     0.0658  0.3409 0.1103
      7 3530        0.0306                  0.0370   0.6572  0.0685       0.0294     0.0594  0.4167 0.1040
      8 2182        0.0371                  0.0364   0.6569  0.0950       0.0350     0.0661  0.

In [46]:
# ============================================
# STEP 45 — Summarize subgroup performance gaps
# ============================================

import pandas as pd

# Ignore very small groups when summarizing gaps.
# Small groups can give unstable metrics.
MIN_GROUP_SIZE = 500


def summarize_group_gap(results, group_column):

    # Keep groups with at least 500 registrations
    stable = results[
        results["N"] >= MIN_GROUP_SIZE
    ].copy()

    # Find lowest and highest recall groups
    min_recall_row = stable.loc[
        stable["Recall"].idxmin()
    ]

    max_recall_row = stable.loc[
        stable["Recall"].idxmax()
    ]

    # Find lowest and highest ROC-AUC groups
    min_auc_row = stable.loc[
        stable["ROC-AUC"].idxmin()
    ]

    max_auc_row = stable.loc[
        stable["ROC-AUC"].idxmax()
    ]

    return {
        "Subgroup": group_column,
        "Lowest Recall Group":
            min_recall_row[group_column],
        "Lowest Recall":
            min_recall_row["Recall"],
        "Highest Recall Group":
            max_recall_row[group_column],
        "Highest Recall":
            max_recall_row["Recall"],
        "Recall Gap":
            max_recall_row["Recall"]
            - min_recall_row["Recall"],
        "Lowest ROC-AUC":
            min_auc_row["ROC-AUC"],
        "Highest ROC-AUC":
            max_auc_row["ROC-AUC"]
    }


# Build one summary table
fairness_summary = pd.DataFrame([
    summarize_group_gap(
        gender_results,
        "GENDER"
    ),

    summarize_group_gap(
        ethnicity_results,
        "ETHCAT"
    ),

    summarize_group_gap(
        abo_results,
        "ABO"
    ),

    summarize_group_gap(
        age_results,
        "AGE_GROUP"
    ),

    summarize_group_gap(
        region_results,
        "REGION"
    )
])

print("FINAL XGBOOST — SUBGROUP GAP SUMMARY")
print(
    f"\nMinimum group size used: {MIN_GROUP_SIZE}\n"
)

print(
    fairness_summary
    .round(4)
    .to_string(index=False)
)

FINAL XGBOOST — SUBGROUP GAP SUMMARY

Minimum group size used: 500

 Subgroup Lowest Recall Group  Lowest Recall Highest Recall Group  Highest Recall  Recall Gap  Lowest ROC-AUC  Highest ROC-AUC
   GENDER                   F         0.4787                    M          0.4843      0.0056          0.6685           0.6832
   ETHCAT                 5.0         0.2414                  1.0          0.5972      0.3558          0.6217           0.6981
      ABO                  AB         0.4143                    A          0.5272      0.1129          0.6499           0.6861
AGE_GROUP                0-17         0.0714                  65+          0.7986      0.7271          0.6209           0.8353
   REGION                 6.0         0.3409                  1.0          0.6222      0.2813          0.6405           0.7409


In [47]:
# ============================================
# STEP 46 — Reliability-aware subgroup summary
# ============================================

import pandas as pd
import numpy as np

# Require enough total registrations AND
# enough actual adverse events for a subgroup
# comparison to be considered reasonably stable
MIN_GROUP_SIZE = 500
MIN_ADVERSE_EVENTS = 30


def subgroup_support(data, group_column):

    # Calculate number of registrations and
    # number of actual adverse events per group
    support = (
        data.groupby(
            group_column,
            dropna=False,
            observed=True
        )["adverse_1yr"]
        .agg(
            N="size",
            Adverse_Count="sum"
        )
        .reset_index()
    )

    return support


def reliable_gap_summary(
    results,
    data,
    group_column
):

    # Get adverse-event counts
    support = subgroup_support(
        data,
        group_column
    )

    # Add counts to existing performance table
    merged = results.merge(
        support,
        on=group_column,
        how="left",
        suffixes=("", "_support")
    )

    # Keep only groups with enough total
    # observations and enough positive cases
    reliable = merged[
        (merged["N"] >= MIN_GROUP_SIZE)
        &
        (
            merged["Adverse_Count"]
            >= MIN_ADVERSE_EVENTS
        )
    ].copy()

    # Find lowest and highest recall
    min_row = reliable.loc[
        reliable["Recall"].idxmin()
    ]

    max_row = reliable.loc[
        reliable["Recall"].idxmax()
    ]

    return {
        "Subgroup": group_column,

        "Lowest Recall Group":
            min_row[group_column],

        "Lowest Recall":
            min_row["Recall"],

        "Lowest Group Adverse Events":
            int(min_row["Adverse_Count"]),

        "Highest Recall Group":
            max_row[group_column],

        "Highest Recall":
            max_row["Recall"],

        "Highest Group Adverse Events":
            int(max_row["Adverse_Count"]),

        "Recall Gap":
            max_row["Recall"]
            - min_row["Recall"]
    }


# Build reliability-aware summary
reliable_fairness_summary = pd.DataFrame([
    reliable_gap_summary(
        gender_results,
        fairness_test,
        "GENDER"
    ),

    reliable_gap_summary(
        ethnicity_results,
        fairness_test,
        "ETHCAT"
    ),

    reliable_gap_summary(
        abo_results,
        fairness_test,
        "ABO"
    ),

    reliable_gap_summary(
        age_results,
        fairness_test,
        "AGE_GROUP"
    ),

    reliable_gap_summary(
        region_results,
        fairness_test,
        "REGION"
    )
])

print(
    "RELIABILITY-AWARE SUBGROUP GAP SUMMARY"
)

print(
    f"\nMinimum registrations: "
    f"{MIN_GROUP_SIZE}"
)

print(
    f"Minimum adverse events: "
    f"{MIN_ADVERSE_EVENTS}\n"
)

print(
    reliable_fairness_summary
    .round(4)
    .to_string(index=False)
)

RELIABILITY-AWARE SUBGROUP GAP SUMMARY

Minimum registrations: 500
Minimum adverse events: 30

 Subgroup Lowest Recall Group  Lowest Recall  Lowest Group Adverse Events Highest Recall Group  Highest Recall  Highest Group Adverse Events  Recall Gap
   GENDER                   F         0.4787                          587                    M          0.4843                          1018      0.0056
   ETHCAT                 5.0         0.2414                           87                  1.0          0.5972                           710      0.3558
      ABO                  AB         0.4143                           70                    A          0.5272                           533      0.1129
AGE_GROUP               18-34         0.0877                           57                  65+          0.7986                           561      0.7109
   REGION                 6.0         0.3409                           44                  1.0          0.6222                            45

In [48]:
# ============================================
# STEP 47 — Subgroup calibration summary
# ============================================

import pandas as pd


def calibration_summary(
    results,
    group_column,
    min_group_size=500
):
    # Keep reasonably large groups
    stable = results[
        results["N"] >= min_group_size
    ].copy()

    # Positive value = model overpredicts risk
    # Negative value = model underpredicts risk
    stable["Calibration Gap"] = (
        stable["Average Predicted Risk"]
        - stable["Adverse Rate"]
    )

    return stable[
        [
            group_column,
            "N",
            "Adverse Rate",
            "Average Predicted Risk",
            "Calibration Gap",
            "Brier Score"
        ]
    ]


# Create calibration summaries
gender_calibration = calibration_summary(
    gender_results,
    "GENDER"
)

ethnicity_calibration = calibration_summary(
    ethnicity_results,
    "ETHCAT"
)

abo_calibration = calibration_summary(
    abo_results,
    "ABO"
)

age_calibration = calibration_summary(
    age_results,
    "AGE_GROUP"
)

region_calibration = calibration_summary(
    region_results,
    "REGION"
)


print("AGE GROUP CALIBRATION\n")

print(
    age_calibration
    .round(4)
    .to_string(index=False)
)

print("\nETHCAT CALIBRATION\n")

print(
    ethnicity_calibration
    .round(4)
    .to_string(index=False)
)

print("\nREGION CALIBRATION\n")

print(
    region_calibration
    .round(4)
    .to_string(index=False)
)

AGE GROUP CALIBRATION

AGE_GROUP     N  Adverse Rate  Average Predicted Risk  Calibration Gap  Brier Score
     0-17  1091        0.0128                  0.0108          -0.0020       0.0124
    18-34  3806        0.0150                  0.0153           0.0003       0.0146
    35-49 10089        0.0241                  0.0249           0.0008       0.0231
    50-64 16722        0.0437                  0.0440           0.0003       0.0413
      65+  9763        0.0575                  0.0650           0.0076       0.0536

ETHCAT CALIBRATION

 ETHCAT     N  Adverse Rate  Average Predicted Risk  Calibration Gap  Brier Score
      1 15984        0.0444                  0.0467           0.0023       0.0417
      2 12080        0.0378                  0.0382           0.0003       0.0361
      4  8985        0.0343                  0.0372           0.0030       0.0320
      5  3036        0.0287                  0.0318           0.0032       0.0276

REGION CALIBRATION

 REGION    N  Adverse

In [49]:
# ============================================
# STEP 48 — Load processed feature names
# for XGBoost explainability
# ============================================

import pandas as pd
import os

# Load the feature names created during
# Model Preparation for the NO-CPRA model
feature_names_df = pd.read_csv(
    os.path.join(
        base_path,
        "features_no_cpra.csv"
    )
)

print("FEATURE FILE LOADED")

print(
    "\nFeature file shape:",
    feature_names_df.shape
)

print(
    "XGBoost input features:",
    X_train.shape[1]
)

print("\nFEATURE FILE COLUMNS")
print(
    feature_names_df.columns.tolist()
)

print("\nFIRST 15 ROWS")
print(
    feature_names_df.head(15).to_string(
        index=False
    )
)

FEATURE FILE LOADED

Feature file shape: (67, 1)
XGBoost input features: 67

FEATURE FILE COLUMNS
['feature']

FIRST 15 ROWS
                                   feature
                         numeric__INIT_AGE
                    numeric__BMI_TCR_clean
                text_categorical__GENDER_F
                text_categorical__GENDER_M
                   text_categorical__ABO_A
                  text_categorical__ABO_A1
                 text_categorical__ABO_A1B
                  text_categorical__ABO_A2
                 text_categorical__ABO_A2B
                  text_categorical__ABO_AB
                   text_categorical__ABO_B
                   text_categorical__ABO_O
           text_categorical__ON_DIALYSIS_N
           text_categorical__ON_DIALYSIS_Y
text_categorical__A2A2B_status_B - missing


In [50]:
# ============================================
# STEP 49 — XGBoost feature importance
# ============================================

import pandas as pd

# Get the 67 processed feature names
feature_names = feature_names_df[
    "feature"
].tolist()

# Get XGBoost feature importance values
# Higher value = the feature contributed more
# to the model's tree decisions
importance_values = (
    xgb_model.feature_importances_
)

# Create a table
feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importance_values
})

# Sort from most important to least important
feature_importance = (
    feature_importance
    .sort_values(
        "Importance",
        ascending=False
    )
    .reset_index(drop=True)
)

print(
    "FINAL XGBOOST — TOP 20 FEATURES\n"
)

print(
    feature_importance
    .head(20)
    .round(4)
    .to_string(index=False)
)

FINAL XGBOOST — TOP 20 FEATURES

                                      Feature  Importance
coded_categorical__FUNC_STAT_TCR_clean_2020.0      0.1528
coded_categorical__FUNC_STAT_TCR_clean_2030.0      0.1035
                            numeric__INIT_AGE      0.0512
coded_categorical__FUNC_STAT_TCR_clean_2080.0      0.0499
coded_categorical__FUNC_STAT_TCR_clean_2010.0      0.0469
coded_categorical__FUNC_STAT_TCR_clean_2090.0      0.0439
coded_categorical__FUNC_STAT_TCR_clean_2100.0      0.0403
coded_categorical__FUNC_STAT_TCR_clean_2040.0      0.0387
coded_categorical__FUNC_STAT_TCR_clean_2050.0      0.0331
text_categorical__A2A2B_status_B - recorded 1      0.0263
              text_categorical__ON_DIALYSIS_Y      0.0213
              text_categorical__ON_DIALYSIS_N      0.0209
                coded_categorical__ETHCAT_1.0      0.0185
                          binary__BMI_missing      0.0162
                coded_categorical__REGION_4.0      0.0149
coded_categorical__FUNC_STAT_TCR_clean_

In [51]:
# ============================================
# STEP 50 — Aggregate XGBoost importance
# back to original variables
# ============================================

import pandas as pd

# Original NO-CPRA variables used in the model
original_features = [
    "INIT_AGE",
    "GENDER",
    "ETHCAT",
    "ABO",
    "ON_DIALYSIS",
    "BMI_TCR_clean",
    "BMI_missing",
    "FUNC_STAT_TCR_clean",
    "FUNC_STAT_missing",
    "INIT_STAT",
    "REGION",
    "A2A2B_status"
]

# Sort longest names first so matching is safer
original_features_sorted = sorted(
    original_features,
    key=len,
    reverse=True
)


def get_original_feature(processed_name):

    # Remove preprocessing prefix such as:
    # numeric__, text_categorical__,
    # coded_categorical__, binary__
    clean_name = processed_name.split(
        "__",
        1
    )[-1]

    # Match processed feature back to
    # its original variable
    for feature in original_features_sorted:

        if (
            clean_name == feature
            or clean_name.startswith(
                feature + "_"
            )
        ):
            return feature

    return "OTHER"


# Add original-variable name
feature_importance[
    "Original Feature"
] = feature_importance[
    "Feature"
].apply(
    get_original_feature
)

# Sum importance across all one-hot
# categories belonging to the same variable
aggregated_importance = (
    feature_importance
    .groupby(
        "Original Feature",
        as_index=False
    )["Importance"]
    .sum()
    .sort_values(
        "Importance",
        ascending=False
    )
    .reset_index(drop=True)
)

# Convert to percentage of total importance
aggregated_importance[
    "Importance %"
] = (
    aggregated_importance["Importance"]
    / aggregated_importance[
        "Importance"
    ].sum()
    * 100
)

print(
    "FINAL XGBOOST — IMPORTANCE "
    "BY ORIGINAL VARIABLE\n"
)

print(
    aggregated_importance
    .round(4)
    .to_string(index=False)
)

FINAL XGBOOST — IMPORTANCE BY ORIGINAL VARIABLE

   Original Feature  Importance  Importance %
FUNC_STAT_TCR_clean      0.5988     59.883701
             REGION      0.0957      9.568000
             ETHCAT      0.0684      6.842900
           INIT_AGE      0.0512      5.120400
       A2A2B_status      0.0456      4.557900
        ON_DIALYSIS      0.0422      4.219900
                ABO      0.0397      3.971800
        BMI_missing      0.0162      1.621100
             GENDER      0.0157      1.571300
          INIT_STAT      0.0139      1.388500
      BMI_TCR_clean      0.0080      0.797300
  FUNC_STAT_missing      0.0046      0.457300


In [52]:
# ============================================
# STEP 51 — SHAP global feature importance
# Final XGBoost model
# ============================================

import shap
import numpy as np
import pandas as pd

# --------------------------------------------
# Use a reproducible sample of 5,000 test
# registrations to keep SHAP fast.
# --------------------------------------------

rng = np.random.RandomState(42)

sample_indices = rng.choice(
    X_test.shape[0],
    size=5000,
    replace=False
)

X_test_shap = X_test[
    sample_indices
]

# --------------------------------------------
# Create SHAP explainer for XGBoost
# --------------------------------------------

explainer = shap.TreeExplainer(
    xgb_model
)

# Calculate SHAP values
shap_values = explainer.shap_values(
    X_test_shap
)

# --------------------------------------------
# Mean absolute SHAP value:
# larger = feature changes predictions more
# --------------------------------------------

mean_abs_shap = np.abs(
    shap_values
).mean(axis=0)

shap_importance = pd.DataFrame({
    "Feature": feature_names,
    "Mean |SHAP|": mean_abs_shap
})

# Match processed features back to
# original variables
shap_importance[
    "Original Feature"
] = shap_importance[
    "Feature"
].apply(
    get_original_feature
)

# Aggregate one-hot columns back to
# original variables
shap_original = (
    shap_importance
    .groupby(
        "Original Feature",
        as_index=False
    )["Mean |SHAP|"]
    .sum()
    .sort_values(
        "Mean |SHAP|",
        ascending=False
    )
    .reset_index(drop=True)
)

# Convert to relative percentage
shap_original[
    "Relative Importance %"
] = (
    shap_original["Mean |SHAP|"]
    / shap_original["Mean |SHAP|"].sum()
    * 100
)

print(
    "FINAL XGBOOST — SHAP IMPORTANCE "
    "BY ORIGINAL VARIABLE\n"
)

print(
    shap_original
    .round(4)
    .to_string(index=False)
)

FINAL XGBOOST — SHAP IMPORTANCE BY ORIGINAL VARIABLE

   Original Feature  Mean |SHAP|  Relative Importance %
FUNC_STAT_TCR_clean       0.4848                35.7719
           INIT_AGE       0.4281                31.5898
             REGION       0.1103                 8.1362
             ETHCAT       0.0888                 6.5516
      BMI_TCR_clean       0.0676                 4.9907
        ON_DIALYSIS       0.0497                 3.6650
       A2A2B_status       0.0416                 3.0692
          INIT_STAT       0.0317                 2.3381
             GENDER       0.0279                 2.0551
                ABO       0.0200                 1.4793
        BMI_missing       0.0044                 0.3236
  FUNC_STAT_missing       0.0004                 0.0294


In [53]:
# ============================================
# STEP 52 — SHAP direction for AGE
# ============================================

import pandas as pd
import numpy as np

# --------------------------------------------
# Find the processed age feature
# --------------------------------------------

age_feature_name = "numeric__INIT_AGE"

age_feature_index = feature_names.index(
    age_feature_name
)

# Get the SHAP value for age
# for the same 5,000 sampled test rows
age_shap_values = shap_values[
    :,
    age_feature_index
]

# --------------------------------------------
# Get the original ages for those same rows
# --------------------------------------------

age_shap_df = pd.DataFrame({
    "INIT_AGE": test_metadata.iloc[
        sample_indices
    ]["INIT_AGE"].to_numpy(),

    "Age_SHAP": age_shap_values
})

# Create age groups
age_shap_df["AGE_GROUP"] = pd.cut(
    age_shap_df["INIT_AGE"],
    bins=[
        -np.inf,
        17,
        34,
        49,
        64,
        np.inf
    ],
    labels=[
        "0-17",
        "18-34",
        "35-49",
        "50-64",
        "65+"
    ]
)

# --------------------------------------------
# Average SHAP effect by age group
#
# Positive SHAP = pushes prediction toward
# higher adverse-event risk
#
# Negative SHAP = pushes prediction toward
# lower adverse-event risk
# --------------------------------------------

age_shap_summary = (
    age_shap_df
    .groupby(
        "AGE_GROUP",
        observed=True
    )
    .agg(
        N=("Age_SHAP", "size"),
        Mean_Age=("INIT_AGE", "mean"),
        Mean_AGE_SHAP=("Age_SHAP", "mean")
    )
    .reset_index()
)

print(
    "FINAL XGBOOST — AGE SHAP DIRECTION\n"
)

print(
    age_shap_summary
    .round(4)
    .to_string(index=False)
)

FINAL XGBOOST — AGE SHAP DIRECTION

AGE_GROUP    N  Mean_Age  Mean_AGE_SHAP
     0-17  129   11.5891        -1.1511
    18-34  447   27.9463        -0.9986
    35-49 1245   42.7534        -0.4419
    50-64 1970   57.3330         0.1025
      65+ 1209   69.3747         0.5209


In [54]:
# ============================================
# STEP 53 — SHAP direction for
# FUNCTIONAL STATUS categories
# ============================================

import pandas as pd
import numpy as np

# --------------------------------------------
# Find all processed features belonging to
# FUNC_STAT_TCR_clean
# --------------------------------------------

func_indices = [
    i
    for i, name in enumerate(feature_names)
    if "FUNC_STAT_TCR_clean_" in name
]

func_rows = []

# --------------------------------------------
# For each functional-status category:
#
# Look only at sampled patients who actually
# belong to that category, then calculate the
# average SHAP effect of that category.
#
# Positive SHAP = pushes risk higher
# Negative SHAP = pushes risk lower
# --------------------------------------------

for idx in func_indices:

    feature_name = feature_names[idx]

    # Get this one-hot column from the
    # sampled test matrix
    active = (
        X_test_shap[:, idx]
        .toarray()
        .ravel()
        > 0
    )

    n_active = active.sum()

    # Skip categories not present in sample
    if n_active == 0:
        continue

    mean_shap = shap_values[
        active,
        idx
    ].mean()

    # Extract category code from feature name
    category = feature_name.split(
        "FUNC_STAT_TCR_clean_",
        1
    )[1]

    func_rows.append({
        "Functional Status Code": category,
        "Sample N": n_active,
        "Mean SHAP": mean_shap
    })


func_shap_summary = pd.DataFrame(
    func_rows
).sort_values(
    "Mean SHAP",
    ascending=False
).reset_index(drop=True)


print(
    "FINAL XGBOOST — FUNCTIONAL STATUS "
    "SHAP DIRECTION\n"
)

print(
    func_shap_summary
    .round(4)
    .to_string(index=False)
)

FINAL XGBOOST — FUNCTIONAL STATUS SHAP DIRECTION

Functional Status Code  Sample N  Mean SHAP
                2010.0         1     2.1807
                2020.0        17     1.7536
                2030.0        21     1.7131
                2040.0        96     0.5996
                4050.0         3     0.4601
                2050.0       257     0.2998
                 998.0        87     0.1459
                4040.0         3     0.1183
                2060.0       370     0.1014
                4060.0         6     0.0918
                2070.0      1091     0.0085
                4070.0        14     0.0078
                 996.0         2     0.0000
                4010.0         1     0.0000
                2080.0      1447    -0.1944
                4090.0        23    -0.2560
                2090.0      1138    -0.3349
                2100.0       347    -0.5530
                -999.0         6    -0.6744
                4080.0        21    -0.7777
                4100.0    

In [55]:
# ============================================
# STEP 54 — SHAP direction for DIALYSIS STATUS
# ============================================

import pandas as pd

# --------------------------------------------
# Find the one-hot features for dialysis
# --------------------------------------------

dialysis_indices = [
    i
    for i, name in enumerate(feature_names)
    if "ON_DIALYSIS_" in name
]

dialysis_rows = []

# --------------------------------------------
# Calculate the SHAP effect for patients
# who actually belong to each dialysis group
#
# Positive SHAP = pushes predicted risk higher
# Negative SHAP = pushes predicted risk lower
# --------------------------------------------

for idx in dialysis_indices:

    feature_name = feature_names[idx]

    # Find sampled patients where this
    # one-hot category is active
    active = (
        X_test_shap[:, idx]
        .toarray()
        .ravel()
        > 0
    )

    n_active = active.sum()

    if n_active == 0:
        continue

    # Average SHAP contribution
    mean_shap = shap_values[
        active,
        idx
    ].mean()

    # Extract Y or N from feature name
    status = feature_name.split(
        "ON_DIALYSIS_",
        1
    )[1]

    dialysis_rows.append({
        "On Dialysis": status,
        "Sample N": n_active,
        "Mean SHAP": mean_shap
    })


dialysis_shap_summary = pd.DataFrame(
    dialysis_rows
).sort_values(
    "On Dialysis"
).reset_index(drop=True)


print(
    "FINAL XGBOOST — DIALYSIS SHAP DIRECTION\n"
)

print(
    dialysis_shap_summary
    .round(4)
    .to_string(index=False)
)

FINAL XGBOOST — DIALYSIS SHAP DIRECTION

On Dialysis  Sample N  Mean SHAP
          N      1684     0.0701
          Y      3316    -0.0070


In [56]:
# ============================================
# STEP 55 — SHAP direction for REGION
# and ETHCAT
# ============================================

import pandas as pd


def categorical_shap_summary(
    feature_keyword,
    label
):

    rows = []

    # Find all one-hot columns for this variable
    indices = [
        i
        for i, name in enumerate(feature_names)
        if feature_keyword in name
    ]

    for idx in indices:

        feature_name = feature_names[idx]

        # Find sampled patients where this
        # one-hot category is active
        active = (
            X_test_shap[:, idx]
            .toarray()
            .ravel()
            > 0
        )

        n_active = active.sum()

        # Skip categories not represented
        if n_active == 0:
            continue

        # Average SHAP contribution
        mean_shap = shap_values[
            active,
            idx
        ].mean()

        category = feature_name.split(
            feature_keyword,
            1
        )[1]

        rows.append({
            label: category,
            "Sample N": n_active,
            "Mean SHAP": mean_shap
        })

    return (
        pd.DataFrame(rows)
        .sort_values(
            "Mean SHAP",
            ascending=False
        )
        .reset_index(drop=True)
    )


# REGION
region_shap_summary = categorical_shap_summary(
    "REGION_",
    "Region"
)

# ETHCAT
ethcat_shap_summary = categorical_shap_summary(
    "ETHCAT_",
    "ETHCAT"
)


print("REGION — SHAP DIRECTION\n")

print(
    region_shap_summary
    .round(4)
    .to_string(index=False)
)

print("\nETHCAT — SHAP DIRECTION\n")

print(
    ethcat_shap_summary
    .round(4)
    .to_string(index=False)
)

REGION — SHAP DIRECTION

Region  Sample N  Mean SHAP
   4.0       603     0.2761
  10.0       342     0.0587
   3.0       662     0.0343
   2.0       532     0.0240
   5.0       807     0.0017
   7.0       462     0.0007
   6.0       139     0.0004
  11.0       664    -0.0057
   9.0       357    -0.0098
   1.0       171    -0.0237
   8.0       261    -0.0764

ETHCAT — SHAP DIRECTION

ETHCAT  Sample N  Mean SHAP
   7.0        21     0.2046
   6.0        35     0.0558
   1.0      1944     0.0426
   9.0        38     0.0062
   2.0      1478     0.0048
 998.0        53     0.0000
   4.0      1086    -0.0087
   5.0       345    -0.1843


In [57]:
# ============================================================
# STEP 56 — MULTI-MODEL COMPARISON
# Logistic Regression, Decision Tree, Random Forest,
# Extra Trees, and XGBoost
#
# IMPORTANT:
# - Train only on TRAIN
# - Compare only on VALIDATION
# - Do NOT touch the test set here
# ============================================================

import time
import pandas as pd
import numpy as np

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier
)

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    precision_recall_curve
)


# ------------------------------------------------------------
# Helper function:
# evaluate probabilities from one model
# ------------------------------------------------------------

def evaluate_model(
    model_name,
    model,
    X_valid,
    y_valid
):

    # Predict probability of adverse outcome
    probabilities = model.predict_proba(
        X_valid
    )[:, 1]

    # ----------------------------------------
    # Threshold-free metrics
    # ----------------------------------------

    roc_auc = roc_auc_score(
        y_valid,
        probabilities
    )

    pr_auc = average_precision_score(
        y_valid,
        probabilities
    )

    brier = brier_score_loss(
        y_valid,
        probabilities
    )

    # ----------------------------------------
    # Find best F1 threshold
    # ----------------------------------------

    precision_values, recall_values, thresholds = (
        precision_recall_curve(
            y_valid,
            probabilities
        )
    )

    precision_t = precision_values[:-1]
    recall_t = recall_values[:-1]

    f1_values = (
        2
        * precision_t
        * recall_t
        / (
            precision_t
            + recall_t
            + 1e-10
        )
    )

    best_index = np.argmax(
        f1_values
    )

    best_f1 = f1_values[
        best_index
    ]

    best_threshold = thresholds[
        best_index
    ]

    # ----------------------------------------
    # Best precision while recall >= 50%
    # ----------------------------------------

    eligible = np.where(
        recall_t >= 0.50
    )[0]

    recall50_index = eligible[
        np.argmax(
            precision_t[eligible]
        )
    ]

    recall50_precision = precision_t[
        recall50_index
    ]

    recall50_recall = recall_t[
        recall50_index
    ]

    recall50_f1 = f1_values[
        recall50_index
    ]

    recall50_threshold = thresholds[
        recall50_index
    ]

    # ----------------------------------------
    # Top-decile lift
    # ----------------------------------------

    ranking = pd.DataFrame({
        "actual": y_valid,
        "risk": probabilities
    }).sort_values(
        "risk",
        ascending=False
    )

    top_n = int(
        np.ceil(
            len(ranking) * 0.10
        )
    )

    top_10 = ranking.iloc[
        :top_n
    ]

    overall_rate = ranking[
        "actual"
    ].mean()

    top_rate = top_10[
        "actual"
    ].mean()

    top_decile_lift = (
        top_rate
        / overall_rate
    )

    capture_rate = (
        top_10["actual"].sum()
        / ranking["actual"].sum()
    )

    return {
        "Model": model_name,
        "ROC-AUC": roc_auc,
        "PR-AUC": pr_auc,
        "Brier": brier,
        "Mean Predicted Risk":
            probabilities.mean(),
        "Best F1": best_f1,
        "Best F1 Threshold":
            best_threshold,
        "Precision @ >=50% Recall":
            recall50_precision,
        "Recall @ >=50% Recall":
            recall50_recall,
        "F1 @ >=50% Recall":
            recall50_f1,
        "Threshold @ >=50% Recall":
            recall50_threshold,
        "Top-Decile Lift":
            top_decile_lift,
        "Top 10% Capture Rate":
            capture_rate
    }


# ------------------------------------------------------------
# Models we want to compare
#
# Logistic Regression and XGBoost are ALREADY trained,
# so we reuse them rather than training again.
# ------------------------------------------------------------

models_to_train = {

    # Simple nonlinear baseline
    "Decision Tree": DecisionTreeClassifier(
        max_depth=8,
        min_samples_leaf=50,
        random_state=42
    ),

    # Bagging model
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=14,
        min_samples_leaf=20,
        max_features="sqrt",
        n_jobs=-1,
        random_state=42
    ),

    # More randomized tree ensemble
    "Extra Trees": ExtraTreesClassifier(
        n_estimators=200,
        max_depth=14,
        min_samples_leaf=20,
        max_features="sqrt",
        n_jobs=-1,
        random_state=42
    )
}


# ------------------------------------------------------------
# Store all results
# ------------------------------------------------------------

all_model_results = []


# ------------------------------------------------------------
# 1. Existing Logistic Regression
# ------------------------------------------------------------

all_model_results.append(
    evaluate_model(
        "Logistic Regression",
        log_reg,
        X_val,
        y_val
    )
)


# ------------------------------------------------------------
# 2. Train and evaluate additional tree models
# ------------------------------------------------------------

trained_extra_models = {}

for model_name, model in models_to_train.items():

    print(
        f"\nTraining {model_name}..."
    )

    start_time = time.time()

    model.fit(
        X_train,
        y_train
    )

    elapsed = (
        time.time()
        - start_time
    )

    print(
        f"{model_name} training complete."
    )

    print(
        f"Training time: "
        f"{elapsed:.1f} seconds"
    )

    trained_extra_models[
        model_name
    ] = model

    result = evaluate_model(
        model_name,
        model,
        X_val,
        y_val
    )

    all_model_results.append(
        result
    )


# ------------------------------------------------------------
# 3. Existing XGBoost
# ------------------------------------------------------------

all_model_results.append(
    evaluate_model(
        "XGBoost",
        xgb_model,
        X_val,
        y_val
    )
)


# ------------------------------------------------------------
# Final comparison table
#
# Sort mainly by PR-AUC because adverse
# outcomes are rare.
# ------------------------------------------------------------

multi_model_comparison = pd.DataFrame(
    all_model_results
)

multi_model_comparison = (
    multi_model_comparison
    .sort_values(
        "PR-AUC",
        ascending=False
    )
    .reset_index(drop=True)
)


print(
    "\n\n=========================================="
)
print(
    "MULTI-MODEL VALIDATION COMPARISON"
)
print(
    "==========================================\n"
)

print(
    multi_model_comparison
    .round(4)
    .to_string(index=False)
)


Training Decision Tree...
Decision Tree training complete.
Training time: 1.6 seconds

Training Random Forest...
Random Forest training complete.
Training time: 149.4 seconds

Training Extra Trees...
Extra Trees training complete.
Training time: 141.1 seconds


MULTI-MODEL VALIDATION COMPARISON

              Model  ROC-AUC  PR-AUC  Brier  Mean Predicted Risk  Best F1  Best F1 Threshold  Precision @ >=50% Recall  Recall @ >=50% Recall  F1 @ >=50% Recall  Threshold @ >=50% Recall  Top-Decile Lift  Top 10% Capture Rate
            XGBoost   0.6855  0.0850 0.0347               0.0404   0.1377             0.0667                    0.0737                 0.5000             0.1284                    0.0510           2.5375                0.2538
Logistic Regression   0.6793  0.0827 0.0348               0.0403   0.1362             0.0799                    0.0730                 0.5021             0.1275                    0.0504           2.4960                0.2497
      Random Forest   0.

In [58]:
# ============================================================
# STEP 57 — SAVE FINAL MODELING OUTPUTS
# ============================================================

import os
import json
import joblib
import pandas as pd

# Create output folder
output_dir = "/content/medimate_classification_outputs"

os.makedirs(
    output_dir,
    exist_ok=True
)


# ------------------------------------------------------------
# 1. Save FINAL primary model
# ------------------------------------------------------------

joblib.dump(
    xgb_model,
    os.path.join(
        output_dir,
        "final_xgboost_model.joblib"
    )
)


# Also save XGBoost's native model format
xgb_model.save_model(
    os.path.join(
        output_dir,
        "final_xgboost_model.json"
    )
)


# ------------------------------------------------------------
# 2. Save Logistic Regression baseline
# ------------------------------------------------------------

joblib.dump(
    log_reg,
    os.path.join(
        output_dir,
        "logistic_regression_baseline.joblib"
    )
)


# ------------------------------------------------------------
# 3. Save Random Forest as strongest
# alternative model
# ------------------------------------------------------------

joblib.dump(
    trained_extra_models["Random Forest"],
    os.path.join(
        output_dir,
        "random_forest_comparison.joblib"
    )
)


# ------------------------------------------------------------
# 4. Save final threshold and model information
# ------------------------------------------------------------

model_settings = {
    "primary_model": "XGBoost",

    "feature_set": "No CPRA",

    "validation_year": 2023,

    "test_year": 2024,

    "validation_selected_threshold":
        float(xgb_recall_threshold),

    "threshold_goal":
        "Approximately 50% recall on validation",

    "test_roc_auc":
        float(test_roc_auc),

    "test_pr_auc":
        float(test_pr_auc),

    "test_brier_score":
        float(test_brier),

    "test_precision":
        float(test_precision_50),

    "test_recall":
        float(test_recall_50),

    "test_f1":
        float(test_f1_50),

    "test_top_decile_lift":
        float(test_top_decile_lift),

    "test_top_10_capture_rate":
        float(test_top_10_capture_rate)
}


with open(
    os.path.join(
        output_dir,
        "final_model_settings.json"
    ),
    "w"
) as f:

    json.dump(
        model_settings,
        f,
        indent=4
    )


# ------------------------------------------------------------
# 5. Save important comparison tables
# ------------------------------------------------------------

multi_model_comparison.to_csv(
    os.path.join(
        output_dir,
        "multi_model_validation_comparison.csv"
    ),
    index=False
)

cpra_final_comparison.to_csv(
    os.path.join(
        output_dir,
        "cpra_sensitivity_comparison.csv"
    ),
    index=False
)

final_generalization_table.to_csv(
    os.path.join(
        output_dir,
        "validation_vs_test_summary.csv"
    ),
    index=False
)

reliable_fairness_summary.to_csv(
    os.path.join(
        output_dir,
        "subgroup_gap_summary.csv"
    ),
    index=False
)


# ------------------------------------------------------------
# 6. Save detailed subgroup results
# ------------------------------------------------------------

gender_results.to_csv(
    os.path.join(
        output_dir,
        "fairness_gender.csv"
    ),
    index=False
)

ethnicity_results.to_csv(
    os.path.join(
        output_dir,
        "fairness_ethcat.csv"
    ),
    index=False
)

abo_results.to_csv(
    os.path.join(
        output_dir,
        "fairness_blood_type.csv"
    ),
    index=False
)

age_results.to_csv(
    os.path.join(
        output_dir,
        "fairness_age.csv"
    ),
    index=False
)

region_results.to_csv(
    os.path.join(
        output_dir,
        "fairness_region.csv"
    ),
    index=False
)


# ------------------------------------------------------------
# 7. Save explainability results
# ------------------------------------------------------------

feature_importance.to_csv(
    os.path.join(
        output_dir,
        "xgboost_processed_feature_importance.csv"
    ),
    index=False
)

aggregated_importance.to_csv(
    os.path.join(
        output_dir,
        "xgboost_original_feature_importance.csv"
    ),
    index=False
)

shap_original.to_csv(
    os.path.join(
        output_dir,
        "shap_original_feature_importance.csv"
    ),
    index=False
)

age_shap_summary.to_csv(
    os.path.join(
        output_dir,
        "shap_age_direction.csv"
    ),
    index=False
)

func_shap_summary.to_csv(
    os.path.join(
        output_dir,
        "shap_functional_status_direction.csv"
    ),
    index=False
)

dialysis_shap_summary.to_csv(
    os.path.join(
        output_dir,
        "shap_dialysis_direction.csv"
    ),
    index=False
)

region_shap_summary.to_csv(
    os.path.join(
        output_dir,
        "shap_region_direction.csv"
    ),
    index=False
)

ethcat_shap_summary.to_csv(
    os.path.join(
        output_dir,
        "shap_ethcat_direction.csv"
    ),
    index=False
)


# ------------------------------------------------------------
# Show saved files
# ------------------------------------------------------------

saved_files = sorted(
    os.listdir(output_dir)
)

print("FINAL MODELING OUTPUTS SAVED\n")

print(
    "Output folder:",
    output_dir
)

print(
    "\nNumber of saved files:",
    len(saved_files)
)

print("\nFILES:")

for file_name in saved_files:
    print("-", file_name)

FINAL MODELING OUTPUTS SAVED

Output folder: /content/medimate_classification_outputs

Number of saved files: 22

FILES:
- cpra_sensitivity_comparison.csv
- fairness_age.csv
- fairness_blood_type.csv
- fairness_ethcat.csv
- fairness_gender.csv
- fairness_region.csv
- final_model_settings.json
- final_xgboost_model.joblib
- final_xgboost_model.json
- logistic_regression_baseline.joblib
- multi_model_validation_comparison.csv
- random_forest_comparison.joblib
- shap_age_direction.csv
- shap_dialysis_direction.csv
- shap_ethcat_direction.csv
- shap_functional_status_direction.csv
- shap_original_feature_importance.csv
- shap_region_direction.csv
- subgroup_gap_summary.csv
- validation_vs_test_summary.csv
- xgboost_original_feature_importance.csv
- xgboost_processed_feature_importance.csv


In [59]:
# ============================================
# STEP 58 — ZIP all classification outputs
# ============================================

import shutil
import os

# Base name for the ZIP file
zip_base = "/content/medimate_classification_modeling_outputs"

# Create ZIP from the saved output folder
zip_path = shutil.make_archive(
    zip_base,
    "zip",
    output_dir
)

print("FINAL CLASSIFICATION OUTPUT PACKAGE CREATED")

print(
    "\nZIP file:",
    zip_path
)

print(
    "File size:",
    round(
        os.path.getsize(zip_path) / (1024 * 1024),
        2
    ),
    "MB"
)

print(
    "\nFiles packaged:",
    len(os.listdir(output_dir))
)

FINAL CLASSIFICATION OUTPUT PACKAGE CREATED

ZIP file: /content/medimate_classification_modeling_outputs.zip
File size: 6.63 MB

Files packaged: 22


## Final Conclusion

In this notebook, I built machine learning models to predict whether a kidney waitlist patient could have a bad outcome within one year. A bad outcome means the patient died or was removed from the waitlist because they became too sick.

I used older years for training, 2023 for validation, and 2024 for the final test. I kept the 2024 test data separate until the end so it could give a fair idea of how the final model performs on new data.

I compared Logistic Regression, Decision Tree, Random Forest, Extra Trees, and XGBoost. XGBoost gave the best overall results, so I selected it as the final model.

On the 2024 test data, XGBoost had:

- ROC-AUC: **0.6742**
- PR-AUC: **0.0860**
- Brier Score: **0.0366**
- Precision: **7.18%**
- Recall: **48.22%**
- F1 Score: **0.1251**

The final threshold was **0.051**. This threshold was selected using the validation data and was not changed after looking at the test results.

The model was also useful for ranking patients by risk. The highest-risk 10% of patients had about **2.53 times** the adverse outcome rate of the full test group. This highest-risk 10% also contained about **25.3% of all adverse outcomes**.

I also tested whether adding CPRA improved the model. The results were almost the same with and without CPRA. Since CPRA was missing for many recent patients, I decided to use the model **without CPRA** as the main model.

The subgroup analysis showed that the model performed very similarly for males and females. However, larger differences were found between age groups, race/ethnicity groups, blood types, and OPTN regions. The biggest difference was between age groups. The model caught a much higher percentage of adverse cases among older patients than among younger patients. These differences should be studied more before the model is used in a real setting.

SHAP analysis was used to understand which features affected the predictions the most. The two most important features were **functional status** and **age**. Older age generally increased the predicted risk, while younger age lowered it. Poorer functional status also generally increased the predicted risk.

### Limitations

- The model does not include every medical or transplant-related factor that may affect patient outcomes.
- Adverse outcomes are rare, so the model has low precision.
- The same threshold does not work equally well for every subgroup.
- CPRA could not be used reliably because it was missing for many recent patients.
- Model performance changed slightly between 2023 and 2024.
- SHAP and feature importance show patterns learned by the model, but they do not prove that a feature causes an outcome.
- 2025 and 2026 were not used for final testing because many patients did not yet have enough one-year follow-up.
- This model should be used for research and risk analysis, not as a direct medical or organ-allocation decision system.

## Final Modeling Decision

**Final model:** XGBoost  
**Feature set:** No CPRA  
**Threshold:** 0.051  
**Main use:** Finding and ranking patients who may have a higher risk of an adverse waitlist outcome.